# TP Traductor: permiso
---

### **Identificación**: (nombre-año-by-apellido. Ej: nossa-2026-by-pandolfo)

`permiso-2026-by-venezia`

### **Objetivo**: (oración que empieza con verbo)

Traducir los roles y permisos de una aplicación web, descritos con palabras simples en español y organizados por sangría, a un módulo JavaScript que controla el acceso en un backend Node.js con Express: una función `puede` que responde si un rol tiene un permiso y un middleware `exigir` que rechaza con **403 (prohibido)** los pedidos sin permiso.

### **Alcance**: (viñetas, describiendo funcionalidades)

**El problema:** en casi toda aplicación web no todos los usuarios pueden hacer lo mismo. En una tienda, un invitado solo mira los productos, un cliente además hace pedidos y un administrador puede cambiar los productos. Si esas reglas se escriben a mano, quedan repartidas en `if` por todo el backend y es fácil olvidarse de una ruta o equivocarse con la herencia (que el administrador también pueda todo lo del cliente) o con las excepciones (un pasante que hace todo lo del administrador menos borrar). **permiso** permite escribir las reglas en un solo lugar, en español, y genera el código que las hace cumplir.

**El traductor hace:**

* Un programa por aplicación: su nombre y uno o más **roles**.
* Cada rol tiene **permisos**: una acción (`ver`, `crear`, `editar`, `borrar` o `todo`, que equivale a las cuatro) sobre un **recurso** con nombre (`'productos'`, `'historias clínicas'`…).
* **Herencia** entre roles (`hereda`): un rol tiene además todos los permisos de otro, también en cadena (admin → cliente → invitado).
* **Excepciones** (`no puede`): un rol pierde una acción que heredaría (*el pasante hereda del admin pero no puede borrar*), o se queda con `todo` menos una acción en el mismo rol. La excepción también se hereda.
* **Estructura por sangría obligatoria** (2 espacios por nivel, como en Python): aplicación → roles → permisos.
* Comentarios de línea con `#`.
* Controles semánticos antes de generar: roles repetidos, herencia de un rol inexistente o de sí mismo, **herencia circular** (directa o indirecta), permisos repetidos, **contradicciones** (`puede ver` y `no puede ver` lo mismo en un rol) y **excepciones de más** (sacar un permiso que el rol nunca tuvo).
* El código JavaScript lo genera un LLM local (ollama) a partir de los datos que arma el parser. El traductor **ejecuta el código generado con Node** y comprueba que, para cada combinación de rol, acción y recurso, responda lo mismo que la tabla de permisos que calculó el parser; si algo no coincide, le pide al LLM que lo corrija.
* El resultado es un archivo `.js` listo para usar en un backend Express:
  ```js
  const { exigir } = require('./tienda.permiso.js');
  app.post('/pedidos', exigir('crear', 'pedidos'), crearPedido);   // invitado → 403, cliente → pasa
  ```
  Si el pedido no trae usuario responde **401** (falta iniciar sesión); si el rol no tiene el permiso, **403**; si lo tiene, deja pasar el pedido. En la sección *Ejecuciones* se levanta un **servidor Express real** con el módulo generado y se le hacen pedidos HTTP.

**El traductor NO hace:**

* Login ni manejo de usuarios: se asume que otra parte del backend ya puso el usuario y su rol en `req.usuario`.
* Base de datos: la tabla de permisos queda escrita en el módulo generado.
* Permisos con condiciones ("solo sus propios pedidos") ni permisos por usuario individual.
* Herencia múltiple (un rol hereda de un solo rol).
* Variables, condicionales, bucles ni funciones: es un lenguaje de dominio específico, no un lenguaje de programación general.

### **Especificaciones léxicas**: (grilla con tokens, lexemas y patrones)

| Lexema | Token | Patrón |
| -- | -- | -- |
| `permisos` | PERMISOS | `permisos` |
| `rol` | ROL | `rol` |
| `hereda` | HEREDA | `hereda` |
| `puede` | PUEDE | `puede` |
| `no` | NO | `no` |
| `ver` | VER | `ver` |
| `crear` | CREAR | `crear` |
| `editar` | EDITAR | `editar` |
| `borrar` | BORRAR | `borrar` |
| `todo` | TODO | `todo` |
| `'Tienda'`, `'cliente'`, `'historias clínicas'` | CADENA | `'[^'\n]*'` |
| fin de línea | NUEVA_LINEA | `\n` |
| aumento de sangría | INDENTA | lo genera el lexer cuando una línea tiene 2 espacios más que la anterior |
| disminución de sangría | DESINDENTA | lo genera el lexer por cada nivel que baja la sangría respecto de la línea anterior |
| `# comentario` | *(se ignora)* | `\#[^\n]*` |
| espacios dentro de la línea | *(se ignora)* | `[ ]+` |

Las palabras reservadas se reconocen con un único patrón de palabra (`[A-Za-zÁÉÍÓÚáéíóúÑñÜü_][A-Za-zÁÉÍÓÚáéíóúÑñÜü_0-9]*`) y una tabla de palabras reservadas; cualquier otra palabra es un error.

**Errores léxicos:** palabra desconocida (por ejemplo `leer` en lugar de `ver`), carácter no reconocido (por ejemplo comillas dobles), cadena sin cerrar, uso de tabulaciones, sangría que no es múltiplo de 2, que sube más de un nivel de una línea a la siguiente o que no coincide con ningún nivel anterior.

### **Especificaciones sintácticas**: (GIC en BNF)

```
<Programa>   ::= PERMISOS CADENA NUEVA_LINEA INDENTA <Roles> DESINDENTA
<Roles>      ::= <Roles> <Rol>
<Roles>      ::= <Rol>
<Rol>        ::= ROL CADENA <Herencia> NUEVA_LINEA INDENTA <Permisos> DESINDENTA
<Herencia>   ::= HEREDA CADENA
<Herencia>   ::= λ
<Permisos>   ::= <Permisos> <Permiso>
<Permisos>   ::= <Permiso>
<Permiso>    ::= PUEDE <Accion> CADENA NUEVA_LINEA
<Permiso>    ::= NO PUEDE <Accion> CADENA NUEVA_LINEA
<Accion>     ::= VER
<Accion>     ::= CREAR
<Accion>     ::= EDITAR
<Accion>     ::= BORRAR
<Accion>     ::= TODO
```

* Símbolos no terminales: `<Programa>`, `<Roles>`, `<Rol>`, `<Herencia>`, `<Permisos>`, `<Permiso>`, `<Accion>`
* Símbolos terminales: los tokens de la grilla léxica
* Axioma: `<Programa>`
* λ representa la cadena vacía: un rol puede no heredar de ningún otro.
* Un permiso puede ser positivo (`puede`) o una excepción (`no puede`); un rol puede tener solo excepciones (por ejemplo, un rol que hereda todo de otro menos algo).
* Producciones recursivas: `<Roles>` y `<Permisos>` → el lenguaje admite infinitos programas fuente.
* `INDENTA` y `DESINDENTA` funcionan como llaves: la gramática exige que los permisos estén dentro de un rol y los roles dentro del programa, e impide un rol dentro de otro rol.

### **Especificaciones semánticas**: (grilla explicando acciones semánticas por cada regla de la GIC)

| Regla | Acciones semánticas |
| -- | -- |
| `<Programa> ::= PERMISOS CADENA NUEVA_LINEA INDENTA <Roles> DESINDENTA` | Arma el módulo JavaScript completo: un comentario con el nombre de la aplicación (la cadena), la constante `PERMISOS` con una entrada por rol, las funciones `contiene` y `puede`, el middleware `exigir` y `module.exports = { PERMISOS, puede, exigir }`. Antes de generar, verifica los controles que necesitan conocer todos los roles (ver abajo). Al final **ejecuta el módulo con Node** y compara todas las combinaciones rol × acción × recurso con la tabla de permisos esperada. |
| `<Roles> ::= <Roles> <Rol>` / `<Rol>` | Junta los roles en el orden en que fueron escritos. **Errores semánticos:** dos roles con el mismo nombre; herencia circular (A hereda de B y B de A, directa o indirectamente: el rol terminaría heredando de sí mismo y `puede` no terminaría nunca); excepción de más (un `no puede` sobre algo que el rol no tiene ni hereda: no cambia nada y seguramente es un error de tipeo en el recurso o en el rol). |
| `<Rol> ::= ROL CADENA <Herencia> NUEVA_LINEA INDENTA <Permisos> DESINDENTA` | Agrega a `PERMISOS` una entrada con el nombre del rol (la cadena) cuyo valor es `{ hereda: ..., puede: { recurso: [acciones] } }`, con sus permisos propios agrupados por recurso; si el rol tiene excepciones, se agrega `noPuede: { recurso: [acciones] }`. **Errores semánticos:** el mismo permiso o excepción repetido dentro del rol; contradicción entre un permiso y una excepción del mismo rol (`puede ver` + `no puede ver`, o cualquier permiso + `no puede todo` sobre el mismo recurso). `puede todo` + `no puede borrar` **no** es contradicción: significa "todo menos borrar". |
| `<Herencia> ::= HEREDA CADENA` | El rol tiene además todos los permisos (y excepciones) del rol nombrado, y los de los roles de los que ese herede, en cadena: `hereda: 'nombre'`. **Errores semánticos:** el rol nombrado no existe; un rol hereda de sí mismo. |
| `<Herencia> ::= λ` | El rol no hereda de nadie: `hereda: null`. |
| `<Permisos> ::= <Permisos> <Permiso>` / `<Permiso>` | Junta los permisos y excepciones del rol en orden. |
| `<Permiso> ::= PUEDE <Accion> CADENA NUEVA_LINEA` | El rol puede hacer la acción sobre el recurso nombrado por la cadena: la acción se agrega al arreglo de ese recurso en `puede`. |
| `<Permiso> ::= NO PUEDE <Accion> CADENA NUEVA_LINEA` | Excepción: el rol **no** puede hacer la acción sobre el recurso aunque la herede (o aunque tenga `todo` sobre ese recurso): la acción se agrega al arreglo de ese recurso en `noPuede`. |
| `<Accion> ::= VER` / `CREAR` / `EDITAR` / `BORRAR` | La acción correspondiente (`'ver'`, `'crear'`, `'editar'`, `'borrar'`). En una API REST suelen corresponder a `GET`, `POST`, `PUT`/`PATCH` y `DELETE`. |
| `<Accion> ::= TODO` | Equivale a las cuatro acciones: se guarda `'todo'` y la función `contiene` lo acepta para cualquier acción. Con `no puede todo`, el rol pierde todas las acciones sobre ese recurso. |

**Cómo se decide un permiso** (regla de permiso): se mira primero el rol pedido; si tiene una excepción que cubre la acción, la respuesta es **no**; si tiene un permiso que la cubre, es **sí**; si no dice nada, se pregunta al rol del que hereda, y así hasta el final de la cadena (**no**). O sea: **lo más cercano al rol gana**, y dentro de un mismo rol la excepción gana sobre el permiso. Por eso un rol que hereda de otro con una excepción también la hereda, pero puede volver a dar ese permiso con su propio `puede`.

**Funciones del módulo** (significado que se le describe al LLM):

* `contiene(lista, accion)`: `true` si la lista de acciones existe y tiene la acción o `'todo'`.
* `puede(rol, accion, recurso)`: aplica la regla de arriba usando `contiene` sobre `noPuede[recurso]` y `puede[recurso]`, siguiendo la cadena `hereda`; si el rol no existe devuelve `false`.
* `exigir(accion, recurso)`: devuelve un middleware de Express `(req, res, next)`: si no hay `req.usuario` responde `401 Falta iniciar sesión`; si `puede(req.usuario.rol, accion, recurso)` llama a `next()`; si no, responde `403 Sin permiso`.

Las verificaciones marcadas como **error semántico** las realiza el traductor antes de generar el código; si hay alguna, no se genera el módulo y se informan todos los errores con su línea. La generación del código de cada regla la realiza el LLM (ollama): a cada pedido se le describe en lenguaje natural el significado de la construcción con los datos del parser (nunca el programa fuente); la respuesta se **ejecuta con Node** para verificarla y, si no cumple, se le devuelve al LLM la lista de problemas para que la corrija: hasta 4 intentos por conversación y, si se traba, hasta 3 conversaciones nuevas con otra semilla.

### **Ejemplo simple**

**Programa de entrada** (`01_minimo.permiso`):
```
permisos 'Blog'
  rol 'lector'
    puede ver 'articulos'
```

**Acciones semánticas aplicadas:**

| Línea | Regla de la GIC | Acción semántica | Resultado |
| -- | -- | -- | -- |
| `permisos 'Blog'` | `<Programa>` | Abre el módulo de permisos de la aplicación `Blog` | comentario `// Permisos de 'Blog'…` y `const PERMISOS = {` |
| `  rol 'lector'` | `<Rol>`, `<Herencia> ::= λ` | Entrada del rol `lector`, que no hereda de nadie | `'lector': { hereda: null, puede: { … } }` |
| `    puede ver 'articulos'` | `<Permiso>`, `<Accion> ::= VER` | El lector puede ver los artículos | `'articulos': ['ver']` |
| (fin del programa) | `<Programa>` | Agrega las funciones y la exportación; verifica con Node | `function contiene`, `function puede`, `function exigir`, `module.exports` |

**Programa traducido** (`01_minimo.permiso.js`, generado por el traductor):
```js
// Permisos de 'Blog': generado por el traductor permiso a partir de 01_minimo.permiso.
// Uso en Express: const { exigir } = require('./este-archivo');
//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);

const PERMISOS = {
  'lector': { hereda: null, puede: { 'articulos': ['ver'] } },
};

function contiene(lista, accion) {
  if (lista === undefined || !Array.isArray(lista)) return false;
  if (lista.includes(accion)) return true;
  if (lista.includes('todo')) return true;
  return false;
}

function puede(rol, accion, recurso) {
  if (!PERMISOS[rol]) return false;
  if (PERMISOS[rol].noPuede && contiene(PERMISOS[rol].noPuede[recurso], accion)) return false;
  if (contiene(PERMISOS[rol].puede[recurso], accion)) return true;
  return puede(PERMISOS[rol].hereda, accion, recurso);
}

function exigir(accion, recurso) {
  return function(req, res, next) {
    if (!req.usuario) {
      return res.status(401).send('Falta iniciar sesión');
    }
    if (puede(req.usuario.rol, accion, recurso)) {
      return next();
    }
    return res.status(403).send('Sin permiso');
  };
}

module.exports = { PERMISOS, puede, exigir };
```

Usado en Express: `app.get('/articulos', exigir('ver', 'articulos'), listarArticulos);` → un `lector` pasa; un usuario con otro rol recibe 403.

### **Ejemplo complejo**

**Programa de entrada** (`02_tienda.permiso`):
```
# Permisos de la tienda online
permisos 'Tienda'
  rol 'invitado'
    puede ver 'productos'
  rol 'cliente' hereda 'invitado'
    puede crear 'pedidos'
    puede ver 'pedidos'
  rol 'admin' hereda 'cliente'
    puede todo 'productos'
    puede borrar 'pedidos'
  # el pasante ayuda al admin, pero no puede borrar nada
  rol 'pasante' hereda 'admin'
    no puede borrar 'productos'
    no puede borrar 'pedidos'
```

**Acciones semánticas que se suman respecto del ejemplo simple:**

| Construcción | Regla de la GIC | Resultado |
| -- | -- | -- |
| `# Permisos de la tienda online` | (comentario) | Se ignora |
| cuatro roles seguidos | `<Roles> ::= <Roles> <Rol>` | Cuatro entradas en `PERMISOS`, en el orden escrito |
| `rol 'cliente' hereda 'invitado'` | `<Herencia> ::= HEREDA CADENA` | `hereda: 'invitado'`: el cliente también puede ver los productos |
| `rol 'admin' hereda 'cliente'` | `<Herencia>` (en cadena) | admin → cliente → invitado: el admin puede crear pedidos (de cliente) y ver productos (de invitado) |
| dos permisos sobre `'pedidos'` | `<Permisos> ::= <Permisos> <Permiso>` | Se agrupan por recurso: `'pedidos': ['crear', 'ver']` |
| `puede todo 'productos'` | `<Accion> ::= TODO` | `'productos': ['todo']`: ver, crear, editar y borrar productos |
| `no puede borrar 'productos'` | `<Permiso> ::= NO PUEDE <Accion> CADENA NUEVA_LINEA` | `noPuede: { productos: ['borrar'] }`: el pasante hereda todo del admin menos borrar |
| (controles) | `<Rol>`, `<Roles>` | Las excepciones del pasante no contradicen ningún permiso propio y sacan algo que sí hereda (del admin), así que son válidas |

**Tabla de permisos generada** (el resto del módulo, `contiene`, `puede` y `exigir`, es igual que en el ejemplo simple):
```js
const PERMISOS = {
  'invitado': { hereda: null, puede: { productos: ['ver'] } },
  'cliente': { hereda: 'invitado', puede: { 'pedidos': ['crear', 'ver'] } },
  admin: { hereda: 'cliente', puede: { productos: ['todo'], pedidos: ['borrar'] } },
  'pasante': { hereda: 'admin', puede: {}, noPuede: { productos: ['borrar'], pedidos: ['borrar'] } },
};
```

**Tabla de permisos resultante** (con la herencia y las excepciones aplicadas; es lo que el traductor verifica con Node):

| Rol | `productos` | `pedidos` |
| -- | -- | -- |
| `invitado` | ver | — |
| `cliente` | ver | ver, crear |
| `admin` | ver, crear, editar, borrar | ver, crear, borrar |
| `pasante` | ver, crear, editar | ver, crear |

**Uso en una API Express** (es el servidor que se levanta en *Ejecuciones*, `permiso/servidor_tienda.js`):
```js
const express = require('express');
const { exigir } = require('./02_tienda.permiso.js');
const app = express();
app.use((req, res, next) => { /* el login pone el usuario y su rol en req.usuario */ next(); });

app.get('/productos',        exigir('ver', 'productos'),    listarProductos);
app.put('/productos/:id',    exigir('editar', 'productos'), editarProducto);   // admin y pasante
app.delete('/productos/:id', exigir('borrar', 'productos'), borrarProducto);   // solo admin
app.post('/pedidos',         exigir('crear', 'pedidos'),    crearPedido);      // cliente, admin y pasante
```

### **Implementación**: (código)

| Elección | Herramienta |
| -- | -- |
| Lenguaje base | Python 3 |
| Metacompilador | `ply.lex` + `ply.yacc` (parser LALR) |
| LLM | ollama con el modelo `llama3.2` (local, sin internet) |
| Verificación del código generado | Node.js (ejecuta el JavaScript del LLM) |
| Demo web | Express 5 (servidor real con el módulo generado) |
| Entorno | Jupyter Notebook |

**Cómo funciona el traductor:**

1. **Análisis léxico** (`lexer_permiso.py`): PLY corta el programa en tokens. Además, un filtro con una **pila de niveles** compara la sangría de cada línea con la anterior y genera los tokens `INDENTA` / `DESINDENTA` (como llaves que abren y cierran bloques).
2. **Análisis sintáctico y semántico** (`parser_permiso.py`): `ply.yacc` reconoce la GIC (sin conflictos) y arma un **árbol** con los roles, sus permisos y sus excepciones. Durante el análisis se hacen los controles semánticos de la grilla, entre ellos la detección de **herencia circular** siguiendo la cadena `hereda` de cada rol, y la de **excepciones de más**, que recorre la herencia para ver si el rol tendría ese permiso. Si hay errores, se informan todos con su línea y **no se llama al LLM**. Con el árbol, el parser también calcula la **tabla de permisos esperada** (herencia, `todo` y excepciones aplicados).
3. **Generación** (`agente_permiso.py`): se recorre el árbol y, por cada rol, se le pide al LLM su entrada de `PERMISOS`, describiéndole en lenguaje natural sus permisos, sus excepciones y de quién hereda. También se le piden las funciones `contiene`, `puede` y `exigir`, describiéndole su significado. La lógica se pide **en partes chicas**: `contiene` resuelve `todo` y `puede` se ocupa solo de la herencia y las excepciones; pedida toda junta, el modelo (de 3B parámetros) arreglaba una parte y rompía otra. Cada respuesta se **limpia y se verifica ejecutándola con Node**: la entrada de cada rol se evalúa y se compara con los datos del parser; `contiene` y `puede` se prueban con tablas de casos conocidos (herencia en cadena, `todo` y excepciones, también heredadas); `exigir` se prueba simulando pedidos de Express (sin usuario, con permiso y sin permiso). Si algo no cumple, se le devuelve al LLM la lista de problemas para que lo corrija (hasta 4 intentos; si se traba en el mismo error, se empieza una conversación nueva con otra semilla, hasta 3 veces). Se usa `temperature=0` para que la salida sea lo más estable posible.
4. **Armado y verificación final** (`traductor_permiso.py`): se unen las partes en un módulo con formato uniforme y se **ejecuta el módulo completo con Node**, comparando `puede` en **todas** las combinaciones rol × acción × recurso (más un rol y un recurso que no existen) con la tabla esperada.

| Archivo | Qué hace |
| -- | -- |
| `permiso/lexer_permiso.py` | Análisis léxico: tokens, palabras reservadas, sangría y errores léxicos |
| `permiso/parser_permiso.py` | Análisis sintáctico (GIC) + controles semánticos; arma el árbol y la tabla esperada |
| `permiso/agente_permiso.py` | Pedidos al LLM, limpieza y verificación de las respuestas con Node |
| `permiso/traductor_permiso.py` | Recorre el árbol, arma el módulo y hace la verificación final |
| `permiso/main.py` | Versión de consola: lee `archivo.permiso` y escribe `archivo.permiso.js` |
| `permiso/servidor_tienda.js` | API Express de ejemplo protegida con el módulo generado (para la demo y un test) |
| `permiso/test_permiso.py` | TestSuite (pytest) |

**Requisitos para ejecutar el notebook:** tener instalado [ollama](https://ollama.com), haber descargado el modelo con `ollama pull llama3.2` y que ollama esté abierto; y tener instalado [Node.js](https://nodejs.org). Para la demo con servidor real, Express (`npm install express`, lo instala la celda de la demo si falta).

In [1]:
%pip install -q ply ollama pytest
import os
os.makedirs('permiso/ejemplos', exist_ok=True)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\Franco\venv-planilla\Scripts\python.exe -m pip install --upgrade pip


In [2]:
%%file permiso/lexer_permiso.py
"""Analizador léxico del lenguaje permiso (PLY).

Además de los tokens comunes, genera tres tokens que marcan la estructura por sangría:
  NUEVA_LINEA  al terminar cada línea con contenido
  INDENTA      cuando una línea tiene 2 espacios más que la anterior (se abre un nivel)
  DESINDENTA   por cada nivel que se cierra cuando la sangría vuelve atrás
"""
import ply.lex as lex

# ---------------------------------------------------------------- palabras reservadas
reservadas = {
    'permisos': 'PERMISOS', 'rol': 'ROL', 'hereda': 'HEREDA', 'puede': 'PUEDE', 'no': 'NO',
    'ver': 'VER', 'crear': 'CREAR', 'editar': 'EDITAR', 'borrar': 'BORRAR', 'todo': 'TODO',
}

tokens = ['CADENA', 'NUEVA_LINEA', 'INDENTA', 'DESINDENTA'] + list(reservadas.values())

ESPACIOS_POR_NIVEL = 2

# Dentro de una línea solo se ignoran espacios (los tabs son error).
t_ignore = ' '


# ---------------------------------------------------------------- reglas con acciones
# PLY prueba las funciones en el orden en que están escritas.

def t_SALTO(t):
    r'(\n[ ]*(\#[^\n]*)?)+'
    # Un salto de línea, junto con las líneas vacías o de solo comentario que lo siguen.
    # El valor que nos interesa es la sangría de la próxima línea con contenido:
    # la cantidad de espacios después del último '\n'.
    t.lexer.lineno += t.value.count('\n')
    ultima = t.value[t.value.rfind('\n') + 1:]
    t.value = len(ultima) - len(ultima.lstrip(' '))
    return t


def t_COMENTARIO(t):
    r'\#[^\n]*'
    pass  # se descarta


def t_CADENA(t):
    r"'[^'\n]*'"
    t.value = t.value[1:-1]  # sin las comillas
    return t


def t_CADENA_SIN_CERRAR(t):
    r"'[^'\n]*"
    t.lexer.errores.append((t.lineno, columna(t), f"Cadena sin cerrar: {t.value}"))


def t_PALABRA(t):
    r'[A-Za-zÁÉÍÓÚáéíóúÑñÜü_][A-Za-zÁÉÍÓÚáéíóúÑñÜü_0-9]*'
    if t.value in reservadas:
        t.type = reservadas[t.value]
        return t
    t.lexer.errores.append((t.lineno, columna(t), f"Palabra desconocida: '{t.value}'"))


def t_error(t):
    c = t.value[0]
    if c == '\t':
        msg = "No se permiten tabulaciones: usá 2 espacios por nivel"
    else:
        msg = f"Carácter no reconocido: '{c}'"
    t.lexer.errores.append((t.lineno, columna(t), msg))
    t.lexer.skip(1)


def columna(t):
    inicio_linea = t.lexer.lexdata.rfind('\n', 0, t.lexpos) + 1
    return t.lexpos - inicio_linea + 1


# ---------------------------------------------------------------- sangría
def _nuevo(tipo, valor, linea):
    t = lex.LexToken()
    t.type, t.value, t.lineno, t.lexpos = tipo, valor, linea, -1
    return t


def _con_sangria(base):
    """Recorre los tokens de PLY y reemplaza cada SALTO por NUEVA_LINEA + INDENTA/DESINDENTA."""
    niveles = [0]                 # pila con la sangría de cada nivel abierto
    ultimo = 'NUEVA_LINEA'        # al empezar, "venimos" de una línea nueva
    while True:
        t = base.token()
        if t is None:
            break
        if t.type != 'SALTO':
            yield t
            ultimo = t.type
            continue

        sangria, linea = t.value, base.lineno
        if ultimo != 'NUEVA_LINEA':                 # cierra la línea anterior
            yield _nuevo('NUEVA_LINEA', None, linea - 1)
            ultimo = 'NUEVA_LINEA'
        if base.lexpos >= len(base.lexdata):
            break                                   # salto final del archivo: no hay más líneas

        actual = niveles[-1]
        if sangria % ESPACIOS_POR_NIVEL != 0:
            base.errores.append((linea, 1, f"Sangría de {sangria} espacios: debe ser múltiplo de {ESPACIOS_POR_NIVEL}"))
        elif sangria > actual:
            if sangria != actual + ESPACIOS_POR_NIVEL:
                base.errores.append((linea, 1, "La sangría sube más de un nivel de una línea a la siguiente"))
            niveles.append(sangria)
            yield _nuevo('INDENTA', None, linea)
        else:
            while sangria < niveles[-1]:
                niveles.pop()
                yield _nuevo('DESINDENTA', None, linea)
            if sangria != niveles[-1]:
                base.errores.append((linea, 1, "La sangría no coincide con ningún nivel anterior"))

    if ultimo != 'NUEVA_LINEA':
        yield _nuevo('NUEVA_LINEA', None, base.lineno)
    while len(niveles) > 1:                         # cierra los niveles abiertos al final
        niveles.pop()
        yield _nuevo('DESINDENTA', None, base.lineno)


class LexerPermiso:
    """Lexer con la misma interfaz que usa ply.yacc (input/token), más la lista de errores."""

    def __init__(self):
        self.base = lex.lex()

    def input(self, texto):
        # Se agrega un '\n' adelante para que la primera línea también pase por la regla de sangría.
        self.base.input('\n' + texto)
        self.base.lineno = 0
        self.base.errores = []
        self._gen = _con_sangria(self.base)

    @property
    def errores(self):
        return self.base.errores

    def token(self):
        return next(self._gen, None)

    def __iter__(self):
        return iter(self.token, None)


tokens = tokens + ['SALTO']  # token interno: nunca llega al parser
lexer = LexerPermiso()

Overwriting permiso/lexer_permiso.py


In [3]:
%%file permiso/parser_permiso.py
"""Analizador sintáctico del lenguaje permiso (ply.yacc) + controles semánticos.

El parser no genera JavaScript: arma un árbol con los roles y sus permisos (diccionarios de Python)
y verifica las reglas semánticas. La generación del código la hace después el agente (LLM),
recorriendo ese árbol, solo si no hubo ningún error.
"""
import ply.yacc as yacc
from lexer_permiso import LexerPermiso, tokens as _tokens_lexer

# SALTO es interno del lexer: el parser nunca lo recibe.
tokens = [t for t in _tokens_lexer if t != 'SALTO']

ACCIONES = ['ver', 'crear', 'editar', 'borrar']   # 'todo' equivale a las cuatro


class ErrorPermiso(Exception):
    """Error de traducción. tipo: 'léxico', 'sintáctico' o 'semántico'. errores: lista de (línea, mensaje)."""

    def __init__(self, tipo, errores):
        self.tipo, self.errores = tipo, errores
        detalle = '\n'.join(f'  línea {l}: {m}' for l, m in errores)
        super().__init__(f'Error {tipo}:\n{detalle}')


_errores_sintacticos = []
_errores_semanticos = []


def error_semantico(linea, mensaje):
    _errores_semanticos.append((linea, mensaje))


# ---------------------------------------------------------------- reglas de la GIC

def p_programa(p):
    'programa : PERMISOS CADENA NUEVA_LINEA INDENTA roles DESINDENTA'
    p[0] = {'aplicacion': p[2], 'roles': p[5]}
    verificar_roles(p[5])


def p_roles(p):
    '''roles : roles rol
             | rol'''
    p[0] = p[1] + [p[2]] if len(p) == 3 else [p[1]]


def p_rol(p):
    'rol : ROL CADENA herencia NUEVA_LINEA INDENTA permisos DESINDENTA'
    rol = {'nombre': p[2], 'hereda': p[3], 'permisos': p[6], 'linea': p.lineno(1)}
    # Un mismo permiso (o excepción) no puede repetirse dentro del rol.
    vistos, repetidos = {}, []
    for permiso in rol['permisos']:
        clave = (permiso['negado'], permiso['accion'], permiso['recurso'])
        if clave in vistos:
            error_semantico(permiso['linea'], f"\"{texto_permiso(permiso)}\" ya está en la línea "
                                              f"{vistos[clave]} del rol '{rol['nombre']}'")
            repetidos.append(permiso)
        else:
            vistos[clave] = permiso['linea']
    # Una excepción no puede contradecir un permiso del mismo rol.
    # Sí vale 'puede todo' + 'no puede <acción>': es "todo menos esa acción".
    for excepcion in (x for x in rol['permisos'] if x['negado'] and x not in repetidos):
        for permiso in (x for x in rol['permisos'] if not x['negado'] and x['recurso'] == excepcion['recurso']):
            if se_contradicen(permiso['accion'], excepcion['accion']):
                error_semantico(excepcion['linea'], f"Contradicción: el rol '{rol['nombre']}' tiene \"{texto_permiso(permiso)}\" "
                                                    f"(línea {permiso['linea']}) y \"{texto_permiso(excepcion)}\"")
    p[0] = rol


def p_herencia(p):
    'herencia : HEREDA CADENA'
    p[0] = p[2]


def p_herencia_vacia(p):
    'herencia : '
    p[0] = None


def p_permisos(p):
    '''permisos : permisos permiso
                | permiso'''
    p[0] = p[1] + [p[2]] if len(p) == 3 else [p[1]]


def p_permiso(p):
    'permiso : PUEDE accion CADENA NUEVA_LINEA'
    p[0] = {'negado': False, 'accion': p[2], 'recurso': p[3], 'linea': p.lineno(1)}


def p_permiso_negado(p):
    'permiso : NO PUEDE accion CADENA NUEVA_LINEA'
    p[0] = {'negado': True, 'accion': p[3], 'recurso': p[4], 'linea': p.lineno(1)}


def se_contradicen(permitida, negada):
    """'puede todo' + 'no puede borrar' es una excepción válida; cualquier otra superposición es contradicción."""
    if negada == 'todo':
        return True                       # no puede nada: choca con cualquier permiso del mismo recurso
    return permitida == negada            # puede ver + no puede ver


def texto_permiso(permiso):
    return f"{'no ' if permiso['negado'] else ''}puede {permiso['accion']} '{permiso['recurso']}'"


def p_accion(p):
    '''accion : VER
              | CREAR
              | EDITAR
              | BORRAR
              | TODO'''
    p[0] = p[1]


# ---------------------------------------------------------------- errores sintácticos

NOMBRES_TOKENS = {
    'NUEVA_LINEA': 'fin de línea', 'INDENTA': 'aumento de sangría',
    'DESINDENTA': 'fin de bloque (menos sangría)',
}


def describir(tok):
    if tok.type == 'CADENA':
        return f"el texto '{tok.value}'"
    return NOMBRES_TOKENS.get(tok.type, f"la palabra '{tok.value}'")


def p_error(tok):
    if tok is None:
        _errores_sintacticos.append((None, 'El programa terminó antes de tiempo (¿falta un rol o un permiso?)'))
    else:
        _errores_sintacticos.append((tok.lineno, f'No se esperaba {describir(tok)}'))
    raise _CortarAnalisis()


class _CortarAnalisis(Exception):
    pass


# ---------------------------------------------------------------- controles semánticos

def verificar_roles(roles):
    """Controles que necesitan conocer todos los roles: nombres repetidos, herencia y excepciones de más."""
    errores_antes = len(_errores_semanticos)
    por_nombre = {}
    for rol in roles:
        if rol['nombre'] in por_nombre:
            error_semantico(rol['linea'], f"El rol '{rol['nombre']}' ya se definió en la línea "
                                          f"{por_nombre[rol['nombre']]['linea']}")
        else:
            por_nombre[rol['nombre']] = rol

    for rol in roles:
        padre = rol['hereda']
        if padre is None:
            continue
        if padre == rol['nombre']:
            error_semantico(rol['linea'], f"El rol '{padre}' no puede heredar de sí mismo")
        elif padre not in por_nombre:
            error_semantico(rol['linea'], f"El rol '{rol['nombre']}' hereda de '{padre}', que no existe")

    # Herencia circular: siguiendo 'hereda' desde un rol se vuelve a ese mismo rol.
    informados = set()
    for rol in roles:
        cadena, actual = [rol['nombre']], rol['hereda']
        while actual in por_nombre and actual not in cadena:
            cadena.append(actual)
            actual = por_nombre[actual]['hereda']
        if actual == rol['nombre'] and len(cadena) > 1 and rol['nombre'] not in informados:
            informados.update(cadena)
            recorrido = ' → '.join(f"'{n}'" for n in cadena + [rol['nombre']])
            error_semantico(rol['linea'], f'Herencia circular: {recorrido}')

    # Excepciones de más: 'no puede' sobre algo que el rol igual no podría hacer.
    # Solo se controla si la herencia está bien (si no, no se puede recorrer la cadena).
    if len(_errores_semanticos) > errores_antes:
        return
    for rol in roles:
        for excepcion in (x for x in rol['permisos'] if x['negado']):
            acciones = ACCIONES if excepcion['accion'] == 'todo' else [excepcion['accion']]
            if not any(tendria_sin_excepcion(por_nombre, rol, a, excepcion['recurso']) for a in acciones):
                error_semantico(excepcion['linea'], f"La excepción \"{texto_permiso(excepcion)}\" no hace falta: "
                                                    f"el rol '{rol['nombre']}' no tiene ese permiso ni lo hereda")


def tendria_sin_excepcion(por_nombre, rol, accion, recurso):
    """¿El rol podría hacer la acción si se ignoraran sus propias excepciones?"""
    propios = [x['accion'] for x in rol['permisos'] if not x['negado'] and x['recurso'] == recurso]
    if accion in propios or 'todo' in propios:
        return True
    return rol['hereda'] is not None and decidir(por_nombre, rol['hereda'], accion, recurso)


# ---------------------------------------------------------------- tabla de permisos esperada

def recursos_de(arbol):
    """Todos los recursos nombrados en el programa, en el orden en que aparecen."""
    recursos = []
    for rol in arbol['roles']:
        for permiso in rol['permisos']:
            if permiso['recurso'] not in recursos:
                recursos.append(permiso['recurso'])
    return recursos


def decidir(por_nombre, nombre, accion, recurso):
    """Regla de permiso: se recorre la cadena de herencia desde el rol hacia arriba y decide el primer rol que
    diga algo sobre esa acción y ese recurso. Dentro de un rol, la excepción ('no puede') gana sobre el permiso."""
    while nombre is not None:
        rol = por_nombre.get(nombre)
        if rol is None:
            return False
        reglas = [x for x in rol['permisos'] if x['recurso'] == recurso and x['accion'] in (accion, 'todo')]
        if any(x['negado'] for x in reglas):
            return False
        if reglas:
            return True
        nombre = rol['hereda']
    return False


def tabla_esperada(arbol):
    """{rol: {recurso: conjunto de acciones permitidas}} con la herencia, 'todo' y las excepciones aplicadas.

    Es lo que el código JavaScript generado tiene que responder: el agente lo usa para verificarlo."""
    por_nombre = {r['nombre']: r for r in arbol['roles']}
    tabla = {}
    for rol in arbol['roles']:
        tabla[rol['nombre']] = {}
        for recurso in recursos_de(arbol):
            acciones = {a for a in ACCIONES if decidir(por_nombre, rol['nombre'], a, recurso)}
            if acciones:
                tabla[rol['nombre']][recurso] = acciones
    return tabla


# ---------------------------------------------------------------- punto de entrada

parser = yacc.yacc(debug=False, write_tables=False)
_lexer = LexerPermiso()


def analizar(texto):
    """Analiza un programa en permiso. Devuelve el árbol de roles o lanza ErrorPermiso."""
    _errores_sintacticos.clear()
    _errores_semanticos.clear()
    arbol = None
    try:
        arbol = parser.parse(texto, lexer=_lexer)
    except _CortarAnalisis:
        pass
    # Se informa primero el error de la etapa más temprana.
    if _lexer.errores:
        raise ErrorPermiso('léxico', [(l, f'{m} (columna {c})') for l, c, m in _lexer.errores])
    if _errores_sintacticos:
        raise ErrorPermiso('sintáctico', list(_errores_sintacticos))
    if _errores_semanticos:
        raise ErrorPermiso('semántico', sorted(_errores_semanticos, key=lambda e: e[0]))
    return arbol

Overwriting permiso/parser_permiso.py


In [4]:
%%file permiso/agente_permiso.py
"""Agente del lenguaje permiso: le pide al LLM (ollama) el código JavaScript de cada parte del módulo de permisos.

Cada pedido describe en lenguaje natural el significado de la construcción (la especificación
semántica) con los datos que armó el parser, nunca el programa fuente. La respuesta se limpia y se
verifica EJECUTÁNDOLA con Node; si tiene problemas, se le devuelve al LLM la lista de problemas para
que la corrija.
"""
import difflib
import json
import re
import shutil
import subprocess

import ollama

MODELO = 'llama3.2'
OPCIONES = {'temperature': 0, 'seed': 7}   # sin azar: misma entrada, misma salida
INTENTOS = 4                               # correcciones dentro de una misma conversación
RONDAS = 3                                 # conversaciones nuevas (con otra semilla) si se agotan los intentos

SISTEMA = ("Sos un programador JavaScript (Node.js). Respondé ÚNICAMENTE con el código pedido: "
           "sin explicaciones, sin comentarios, sin markdown y sin ```. "
           "No uses require, import ni console.log, y no agregues nada que no se haya pedido.")


class ErrorAgente(Exception):
    pass


# ---------------------------------------------------------------- Node

NODE = shutil.which('node')


def ejecutar_node(codigo):
    """Ejecuta código JavaScript con Node. Devuelve (salió bien, salida o mensaje de error)."""
    if NODE is None:
        raise ErrorAgente('No se encontró Node.js: instalalo desde https://nodejs.org')
    r = subprocess.run([NODE, '-'], input=codigo, capture_output=True, text=True, encoding='utf-8', timeout=20)
    if r.returncode != 0:
        lineas = [l for l in r.stderr.splitlines() if re.match(r'\w*Error\b', l.strip())]
        return False, lineas[0].strip() if lineas else r.stderr.strip()[-300:]
    return True, r.stdout


# ---------------------------------------------------------------- comunicación con el LLM

def error_de_ejecucion(salida, codigo):
    """Mensaje para el LLM cuando su código falla. Si usó un nombre que no existe, sugiere el más parecido."""
    mensaje = f'el código da error al ejecutarlo: {salida}'
    m = re.search(r'ReferenceError: (\w+) is not defined', salida)
    if m:
        nombres = set(re.findall(r'[A-Za-z_]\w*', codigo)) - {m.group(1)}
        parecido = difflib.get_close_matches(m.group(1), nombres, n=1)
        if parecido:
            mensaje += f" (usás '{m.group(1)}', que no existe: ¿quisiste escribir '{parecido[0]}'?)"
    return mensaje


def limpiar(respuesta):
    """Saca lo que el LLM agrega de más: bloques ```, texto antes del código y tabulaciones."""
    bloques = re.findall(r'```[a-zA-Z]*\n(.*?)```', respuesta, flags=re.S)
    codigo = bloques[0] if bloques else re.sub(r'```[a-zA-Z]*', '', respuesta)
    codigo = codigo.strip()
    if len(codigo) > 1 and codigo[0] == codigo[-1] == '"':   # a veces devuelve el código entre comillas, como texto
        codigo = codigo[1:-1]
    lineas = [l.rstrip().replace('\t', '  ') for l in codigo.strip().splitlines()]
    return '\n'.join(l for l in lineas if not l.strip().startswith(('console.log', 'module.exports')))


def extraer_funcion(codigo, nombre):
    """Se queda solo con la declaración 'function nombre(...) { ... }' (el LLM a veces agrega
    constantes o ejemplos de uso alrededor). Si no la encuentra, devuelve el código tal cual."""
    m = re.search(rf'function\s+{nombre}\s*\(', codigo)
    if m is None:
        return codigo
    profundidad, i = 0, codigo.find('{', m.end())
    while 0 <= i < len(codigo):
        profundidad += {'{': 1, '}': -1}.get(codigo[i], 0)
        if profundidad == 0:
            return codigo[m.start():i + 1]
        i += 1
    return codigo[m.start():]


_memoria = {}  # pedido -> respuesta ya verificada (evita repetir pedidos idénticos)


def pedir(pedido, verificar, funcion=None):
    """Hace el pedido, verifica la respuesta y, si hay problemas, pide la corrección.
    funcion: si se pidió una función, nombre de la función que hay que extraer de la respuesta."""
    if pedido not in _memoria:
        _memoria[pedido] = _pedir_al_llm(pedido, verificar, funcion)
    return _memoria[pedido]


def _pedir_al_llm(pedido, verificar, funcion):
    problemas = []
    for ronda in range(RONDAS):
        # Si una conversación se traba en el mismo error, se empieza otra desde cero con otra semilla.
        codigo, problemas = _conversacion(pedido, verificar, funcion, {**OPCIONES, 'seed': OPCIONES['seed'] + ronda})
        if not problemas:
            return codigo
    raise ErrorAgente(f'El LLM no generó código válido después de {RONDAS} rondas de {INTENTOS} intentos: '
                      f'{"; ".join(problemas)}')


def _conversacion(pedido, verificar, funcion, opciones):
    mensajes = [{'role': 'system', 'content': SISTEMA}, {'role': 'user', 'content': pedido}]
    for _ in range(INTENTOS):
        respuesta = ollama.chat(model=MODELO, messages=mensajes, options=opciones)['message']['content']
        codigo = limpiar(respuesta)
        if funcion:
            codigo = extraer_funcion(codigo, funcion)
        problemas = verificar(codigo)
        if not problemas:
            return codigo, []
        mensajes += [
            {'role': 'assistant', 'content': respuesta},
            {'role': 'user', 'content': 'Tu respuesta tiene estos problemas:\n- ' + '\n- '.join(problemas)
                                        + '\nCorregilos y devolvé de nuevo el código COMPLETO, sin explicaciones.'},
        ]
    return codigo, problemas


# ---------------------------------------------------------------- <Rol>: una entrada de la tabla PERMISOS

def texto_js(cadena):
    """Una cadena como literal de JavaScript con comillas simples: cliente -> 'cliente'."""
    return "'" + cadena.replace('\\', '\\\\').replace("'", "\\'") + "'"


def acciones_por_recurso(rol, negado=False):
    """Permisos propios del rol (o sus excepciones, si negado) agrupados por recurso, en el orden escrito."""
    agrupado = {}
    for permiso in rol['permisos']:
        if permiso['negado'] == negado:
            agrupado.setdefault(permiso['recurso'], []).append(permiso['accion'])
    return agrupado


def detalle(agrupado, verbo):
    return '\n'.join(f'- sobre el recurso {texto_js(r)} {verbo}: ' + ', '.join(texto_js(a) for a in acciones)
                     for r, acciones in agrupado.items())


def pedido_rol(rol):
    propios, excepciones = acciones_por_recurso(rol), acciones_por_recurso(rol, negado=True)
    herencia = (f'hereda los permisos del rol {texto_js(rol["hereda"])}: la propiedad hereda vale {texto_js(rol["hereda"])}'
                if rol['hereda'] else 'no hereda de ningún rol: la propiedad hereda vale null')
    lineas = [f'Generá UNA propiedad de un objeto JavaScript que describe los permisos del rol {texto_js(rol["nombre"])}.',
              f'La clave es el nombre del rol entre comillas simples: {texto_js(rol["nombre"])}.',
              'El valor es un objeto con exactamente ' + ('tres propiedades: hereda, puede y noPuede.' if excepciones
                                                          else 'dos propiedades: hereda y puede.'),
              f'- El rol {herencia}.',
              '- puede es un objeto: cada clave es un recurso (entre comillas simples) y su valor es un arreglo '
              'con los nombres de las acciones permitidas sobre ese recurso (entre comillas simples).'
              + (' Este rol NO tiene permisos propios: puede es un objeto vacío, puede: {}.' if not propios else '')]
    if excepciones:
        lineas.append('- noPuede tiene la misma forma que puede, pero con las acciones que el rol NO puede hacer '
                      'aunque las herede (excepciones).')
    if propios:
        lineas.append(f'Permisos propios del rol:\n{detalle(propios, "puede")}')
    if excepciones:
        lineas.append(f'Excepciones del rol (van en noPuede):\n{detalle(excepciones, "no puede")}')
    forma = (f'{texto_js(rol["nombre"])}: {{ hereda: ..., puede: ' + ('{ ... }' if propios else '{}')
             + (', noPuede: { ... }' if excepciones else '') + ' },')
    lineas.append(f'Forma de la respuesta (completá los ...): {forma}')
    lineas.append('Escribí solamente esa propiedad, en una sola línea, terminada en coma, sin encerrarla entre comillas. '
                  'No escribas const, ni el objeto que la contiene, ni llaves de más.')
    return '\n'.join(lineas)


def comparar_acciones(obtenido, esperado, propiedad):
    """Compara {recurso: [acciones]} del LLM con lo esperado (sin importar el orden de las acciones)."""
    if not isinstance(obtenido, dict) or not all(isinstance(a, list) for a in obtenido.values()):
        return [f'{propiedad} tiene que ser un objeto cuyos valores sean arreglos de acciones']
    obtenido = {r: sorted(a) for r, a in obtenido.items()}
    if not esperado and obtenido:
        return [f'{propiedad} tiene que ser exactamente {{}} (un objeto vacío): este rol no tiene nada propio ahí; '
                f'sacá {", ".join(texto_js(r) for r in obtenido)}']
    p = []
    for recurso, acciones in esperado.items():
        if obtenido.get(recurso) != sorted(acciones):
            p.append(f'en {propiedad}, el recurso {texto_js(recurso)} tiene que tener exactamente las acciones '
                     + '[' + ', '.join(texto_js(a) for a in acciones) + ']')
    for recurso in obtenido.keys() - esperado.keys():
        p.append(f'en {propiedad} sobra el recurso {texto_js(recurso)}: no se pidió')
    return p


def verificar_rol(codigo, rol):
    clave = texto_js(rol['nombre'])
    if '\n' in codigo.strip():
        return ['escribí la propiedad en UNA sola línea']
    if re.match(r'\s*(const|let|var)\b', codigo):
        return ['escribí solo la propiedad, sin const/let/var ni el objeto que la contiene']
    if re.match(r'\s*hereda\s*:', codigo):
        return [f'falta la clave del rol: la propiedad tiene que empezar con {clave}: y su valor es el objeto '
                f'{{ hereda: ..., puede: ... }}']
    ok, salida = ejecutar_node(f'const t = {{\n{codigo}\n}};\nprocess.stdout.write(JSON.stringify(t));')
    if not ok:
        return [f'no es una propiedad de objeto JavaScript válida ({salida})']
    tabla = json.loads(salida)
    nombre = rol['nombre']
    if list(tabla) != [nombre]:
        return [f'tiene que haber una sola propiedad, con la clave {texto_js(nombre)} (hay: {", ".join(tabla) or "ninguna"})']
    valor = tabla[nombre]
    excepciones = acciones_por_recurso(rol, negado=True)
    claves = {'hereda', 'puede', 'noPuede'} if excepciones else {'hereda', 'puede'}
    if not isinstance(valor, dict) or set(valor) != claves:
        return ['el valor tiene que ser un objeto con exactamente las propiedades ' + ', '.join(sorted(claves))]
    p = []
    if valor['hereda'] != rol['hereda']:
        p.append(f'hereda tiene que valer {texto_js(rol["hereda"]) if rol["hereda"] else "null"}')
    p += comparar_acciones(valor['puede'], acciones_por_recurso(rol), 'puede')
    if excepciones:
        p += comparar_acciones(valor['noPuede'], excepciones, 'noPuede')
    return p


def generar_rol(rol):
    return pedir(pedido_rol(rol), lambda c: verificar_rol(c, rol))


# ---------------------------------------------------------------- <Accion>: la función contiene

PEDIDO_CONTIENE = (
    'Escribí la función JavaScript contiene(lista, accion), que dice si una lista de acciones permite una acción.\n'
    'lista es un arreglo de textos (por ejemplo [\'ver\', \'crear\']) o puede ser undefined.\n'
    'Reglas:\n'
    '1. Si lista es undefined (o no es un arreglo), devuelve false.\n'
    '2. Si lista contiene la acción pedida (lista.includes(accion)), devuelve true.\n'
    "3. Si lista contiene el texto 'todo', devuelve true: 'todo' significa cualquier acción. "
    "'todo' es un valor guardado en el arreglo: se busca con lista.includes('todo').\n"
    '4. En cualquier otro caso devuelve false.\n'
    'La función devuelve siempre true o false. Escribí solo la función, con la palabra function.'
)

# (acciones, accion, resultado esperado); None representa undefined
CASOS_CONTIENE = [
    (['ver'], 'ver', True), (['ver', 'crear'], 'crear', True), (['ver'], 'borrar', False),
    (['todo'], 'editar', True), (['crear', 'todo'], 'ver', True), ([], 'ver', False), (None, 'ver', False),
]


def comparacion_con_todo(codigo):
    """Error típico del LLM: comparar el arreglo entero con el texto 'todo' (un arreglo nunca es igual a un texto)."""
    if re.search(r"(\]|lista)\s*===?\s*['\"]todo['\"]", codigo):
        return ["en tu código comparás un arreglo con 'todo' usando ===: eso siempre da false porque 'todo' está "
                "DENTRO del arreglo; usá .includes('todo')"]
    return []


def verificar_contiene(codigo):
    if not re.search(r'function\s+contiene\s*\(', codigo):
        return ['tiene que ser una función declarada como function contiene(lista, accion)']
    casos = [[a if a is not None else '__undefined__', x] for a, x, _ in CASOS_CONTIENE]
    prueba = (f'{codigo}\nconst casos = {json.dumps(casos)};\n'
              'process.stdout.write(JSON.stringify(casos.map(([a, x]) => contiene(a === "__undefined__" ? undefined : a, x))));')
    ok, salida = ejecutar_node(prueba)
    if not ok:
        return [error_de_ejecucion(salida, codigo)]
    p = comparacion_con_todo(codigo)
    for (acciones, accion, esperado), obtenido in zip(CASOS_CONTIENE, json.loads(salida)):
        if obtenido is not esperado:
            arg = 'undefined' if acciones is None else json.dumps(acciones).replace('"', "'")
            p.append(f"contiene({arg}, '{accion}') devolvió {json.dumps(obtenido)} y tenía que devolver {json.dumps(esperado)}")
    return p


def generar_contiene():
    return pedir(PEDIDO_CONTIENE, verificar_contiene, funcion='contiene')


# ---------------------------------------------------------------- <Herencia> y excepciones: la función puede

# Tabla de prueba con herencia en cadena, 'todo' y excepciones, para verificar la función con casos conocidos.
TABLA_PRUEBA = {
    'base':     {'hereda': None, 'puede': {'notas': ['ver']}},
    'medio':    {'hereda': 'base', 'puede': {'fotos': ['crear', 'borrar']}},
    'alto':     {'hereda': 'medio', 'puede': {'notas': ['todo']}},
    'limitado': {'hereda': 'alto', 'puede': {}, 'noPuede': {'notas': ['borrar'], 'fotos': ['todo']}},
    'nieto':    {'hereda': 'limitado', 'puede': {}},
    'casi':     {'hereda': 'base', 'puede': {'fotos': ['todo']}, 'noPuede': {'fotos': ['borrar']}},
}
CASOS_PRUEBA = [
    ('base', 'ver', 'notas', True), ('base', 'editar', 'notas', False), ('base', 'crear', 'fotos', False),
    ('medio', 'ver', 'notas', True), ('medio', 'crear', 'fotos', True), ('medio', 'editar', 'fotos', False),
    ('alto', 'editar', 'notas', True), ('alto', 'borrar', 'notas', True), ('alto', 'borrar', 'fotos', True),
    ('alto', 'ver', 'fotos', False), ('nadie', 'ver', 'notas', False), ('base', 'ver', 'otra', False),
    ('limitado', 'editar', 'notas', True), ('limitado', 'borrar', 'notas', False),
    ('limitado', 'crear', 'fotos', False), ('nieto', 'borrar', 'notas', False), ('nieto', 'ver', 'notas', True),
    ('casi', 'editar', 'fotos', True), ('casi', 'borrar', 'fotos', False), ('casi', 'ver', 'notas', True),
]

# contiene() correcta, para probar puede() aislada de los errores que pudiera tener la del LLM.
CONTIENE_REFERENCIA = ("function contiene(lista, accion) {\n"
                       "  return Array.isArray(lista) && (lista.includes(accion) || lista.includes('todo'));\n}")

PEDIDO_PUEDE = (
    'Escribí la función JavaScript puede(rol, accion, recurso), que decide si un rol tiene permiso.\n'
    'Usa la constante global PERMISOS, que YA ESTÁ DEFINIDA afuera de la función: no la declares ni la copies. '
    'Para cada rol, PERMISOS[rol].hereda es el nombre del rol del que hereda (o null); PERMISOS[rol].puede es un '
    'objeto que a cada recurso le asigna un arreglo con las acciones permitidas; y algunos roles tienen además '
    'PERMISOS[rol].noPuede, con la misma forma, que lista excepciones: acciones que el rol NO puede hacer aunque '
    'las herede. Los roles sin excepciones no tienen la propiedad noPuede.\n'
    'Para saber si un arreglo de acciones incluye la acción, usá la función contiene(lista, accion), que YA ESTÁ '
    'DEFINIDA (tiene en cuenta \'todo\' y devuelve false si el arreglo es undefined): no la escribas.\n'
    'Reglas, que se aplican primero al rol pedido:\n'
    '1. Si el rol no existe en PERMISOS, devuelve false.\n'
    '2. Si el rol tiene noPuede y contiene(PERMISOS[rol].noPuede[recurso], accion) es true, devuelve false: '
    'la excepción gana.\n'
    '3. Si contiene(PERMISOS[rol].puede[recurso], accion) es true, devuelve true.\n'
    '4. Si no, aplica las mismas reglas al rol indicado en PERMISOS[rol].hereda, y así sucesivamente.\n'
    '5. Si llega a un rol cuyo hereda es null sin decidir, devuelve false.\n'
    'La función devuelve siempre true o false, nunca undefined ni null.\n'
    'Escribí solo la función puede, con la palabra function.'
)


def explicar(tabla, rol, accion, recurso):
    """Explica con los datos de la tabla de prueba por qué puede(rol, accion, recurso) da lo que da."""
    if rol not in tabla:
        return f"el rol '{rol}' no existe en PERMISOS"
    pasos, actual = [], rol
    while actual is not None:
        negadas = tabla[actual].get('noPuede', {}).get(recurso)
        if negadas is not None and (accion in negadas or 'todo' in negadas):
            pasos.append(f"el arreglo noPuede de '{recurso}' del rol '{actual}' es {json.dumps(negadas)} (excepción)")
            return '; '.join(pasos)
        acciones = tabla[actual]['puede'].get(recurso)
        if acciones is None:
            pasos.append(f"el rol '{actual}' no tiene el recurso '{recurso}' en puede")
        else:
            pasos.append(f"el arreglo puede de '{recurso}' del rol '{actual}' es {json.dumps(acciones)}")
            if accion in acciones or 'todo' in acciones:
                return '; '.join(pasos)
        siguiente = tabla[actual]['hereda']
        pasos.append(f"hereda de '{siguiente}'" if siguiente else 'no hereda de nadie')
        actual = siguiente
    return '; '.join(pasos)


def verificar_puede(codigo, tabla=TABLA_PRUEBA, casos=CASOS_PRUEBA):
    if not re.search(r'function\s+puede\s*\(', codigo):
        return ['tiene que ser una función declarada como function puede(rol, accion, recurso)']
    if re.search(r'\bPERMISOS\s*=', codigo):
        return ['no declares PERMISOS adentro de la función: ya está definida afuera con los datos reales; usala tal cual']
    prueba = (f'const PERMISOS = {json.dumps(tabla)};\n{CONTIENE_REFERENCIA}\n{codigo}\n'
              f'const casos = {json.dumps(casos)};\n'
              'process.stdout.write(JSON.stringify(casos.map(([r, a, x]) => puede(r, a, x))));')
    ok, salida = ejecutar_node(prueba)
    if not ok:
        return [error_de_ejecucion(salida, codigo)]
    p = comparacion_con_todo(codigo)
    for (rol, accion, recurso, esperado), obtenido in zip(casos, json.loads(salida)):
        if obtenido is not esperado:
            p.append(f"puede('{rol}', '{accion}', '{recurso}') devolvió {json.dumps(obtenido)} y tenía que devolver "
                     f'{json.dumps(esperado)} porque {explicar(tabla, rol, accion, recurso)}')
    return p[:5]


def generar_puede():
    return pedir(PEDIDO_PUEDE, verificar_puede, funcion='puede')


# ---------------------------------------------------------------- middleware de Express: la función exigir

PEDIDO_EXIGIR = (
    'Escribí la función JavaScript exigir(accion, recurso) para usar como middleware de Express, '
    "por ejemplo: app.post('/pedidos', exigir('crear', 'pedidos'), crearPedido).\n"
    'exigir devuelve una función (req, res, next) que hace lo siguiente:\n'
    "1. Si req.usuario no existe, responde con estado 401 y el texto 'Falta iniciar sesión': "
    "return res.status(401).send('Falta iniciar sesión').\n"
    "2. Si puede(req.usuario.rol, accion, recurso) es true, llama a next() para que el pedido siga.\n"
    "3. Si no, responde con estado 403 y el texto 'Sin permiso': return res.status(403).send('Sin permiso').\n"
    'La función puede ya está definida: no la escribas. Escribí solo la función exigir, con la palabra function.'
)

# (usuario, permitido) -> resultado esperado del middleware
CASOS_EXIGIR = [
    (None, False, ['estado', 401, 'Falta iniciar sesión']),
    ({'rol': 'cliente'}, True, ['siguiente']),
    ({'rol': 'invitado'}, False, ['estado', 403, 'Sin permiso']),
]


def verificar_exigir(codigo):
    if not re.search(r'function\s+exigir\s*\(', codigo):
        return ['tiene que ser una función declarada como function exigir(accion, recurso)']
    if re.search(r'function\s+puede\s*\(', codigo):
        return ['no escribas la función puede: ya está definida']
    # Se simula Express: un req con o sin usuario, un res que anota lo que se responde y un next.
    prueba = (f"let permitido = false;\nfunction puede(rol, accion, recurso) {{ return permitido; }}\n{codigo}\n"
              f'const casos = {json.dumps(CASOS_EXIGIR)};\nconst resultados = [];\n'
              'for (const [usuario, ok] of casos) {\n'
              '  permitido = ok;\n'
              '  let r = null;\n'
              '  const res = { status(n) { return { send(t) { r = ["estado", n, t]; } }; } };\n'
              '  exigir("crear", "pedidos")(usuario ? { usuario } : {}, res, () => { r = ["siguiente"]; });\n'
              '  resultados.push(r);\n'
              '}\n'
              'process.stdout.write(JSON.stringify(resultados));')
    ok, salida = ejecutar_node(prueba)
    if not ok:
        return [error_de_ejecucion(salida, codigo)]
    p = []
    descripciones = ['sin usuario (req.usuario no existe)', 'con un usuario que tiene permiso',
                     'con un usuario que no tiene permiso']
    for desc, (_, _, esperado), obtenido in zip(descripciones, CASOS_EXIGIR, json.loads(salida)):
        if obtenido != esperado:
            quiere = 'llamar a next()' if esperado[0] == 'siguiente' else f"responder res.status({esperado[1]}).send('{esperado[2]}')"
            p.append(f'{desc} tiene que {quiere}')
    return p


def generar_exigir():
    return pedir(PEDIDO_EXIGIR, verificar_exigir, funcion='exigir')

Overwriting permiso/agente_permiso.py


In [5]:
%%file permiso/traductor_permiso.py
"""Traductor permiso: programa fuente -> módulo JavaScript de permisos.

1. Análisis (lexer + parser + controles semánticos): arma el árbol o informa errores.
2. Generación: recorre el árbol y le pide al agente (LLM) el código de cada rol y de las funciones.
3. Armado y verificación final: une las partes y ejecuta el módulo completo con Node, comparando
   TODAS las combinaciones de rol, acción y recurso con la tabla que calculó el parser.
"""
import json
import re

import agente_permiso as agente
from parser_permiso import analizar, tabla_esperada, recursos_de, ACCIONES

SANGRIA = '  '


def traducir(texto, avisar=lambda mensaje: None, origen='programa'):
    """Traduce un programa en permiso a un módulo JavaScript. Lanza ErrorPermiso si el programa tiene errores."""
    arbol = analizar(texto)
    avisar(f"permisos '{arbol['aplicacion']}'")
    entradas = []
    for rol in arbol['roles']:
        excepciones = sum(p['negado'] for p in rol['permisos'])
        avisar(f"  rol '{rol['nombre']}'" + (f" (hereda de '{rol['hereda']}')" if rol['hereda'] else '')
               + (f" con {excepciones} {'excepciones' if excepciones > 1 else 'excepción'}" if excepciones else ''))
        entradas.append(agente.generar_rol(rol))
    avisar('  función contiene')
    contiene = agente.generar_contiene()
    avisar('  función puede')
    puede = agente.generar_puede()
    avisar('  middleware exigir')
    exigir = agente.generar_exigir()
    modulo = armar_modulo(arbol, origen, entradas, [contiene, puede, exigir])
    avisar('  verificación final con Node')
    problemas = verificar_modulo(modulo, arbol)
    if problemas:
        raise agente.ErrorAgente('El módulo generado no respeta la tabla de permisos: ' + '; '.join(problemas))
    return modulo


# ---------------------------------------------------------------- armado

def armar_modulo(arbol, origen, entradas, funciones):
    lineas = [f"// Permisos de '{arbol['aplicacion']}': generado por el traductor permiso a partir de {origen}.",
              '// Uso en Express: const { exigir } = require(\'./este-archivo\');',
              "//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);",
              '',
              'const PERMISOS = {']
    lineas += [SANGRIA + formatear_entrada(e) for e in entradas]
    lineas.append('};')
    for funcion in funciones:
        lineas += ['', funcion.strip()]
    lineas += ['', 'module.exports = { PERMISOS, puede, exigir };']
    return '\n'.join(lineas) + '\n'


def formatear_entrada(entrada):
    """Espaciado uniforme en las llaves ({ a } en lugar de {a}, {} si está vacío) y una coma al final de cada entrada."""
    e = re.sub(r'\{\s*', '{ ', entrada.strip().rstrip(','))
    e = re.sub(r'\s*\}', ' }', e)
    e = e.replace('{  }', '{}').replace('{ }', '{}')   # objeto vacío
    return e + ','


# ---------------------------------------------------------------- verificación final

def combinaciones(arbol):
    """Todas las preguntas rol × acción × recurso, más un recurso y un rol que no existen."""
    roles = [r['nombre'] for r in arbol['roles']] + ['rol que no existe']
    recursos = recursos_de(arbol) + ['recurso que no existe']
    return [(r, a, x) for r in roles for a in ACCIONES for x in recursos]


def verificar_modulo(modulo, arbol):
    tabla = tabla_esperada(arbol)
    casos = combinaciones(arbol)
    prueba = (modulo.replace('module.exports', 'const _exportado') +
              f'\nconst casos = {json.dumps(casos, ensure_ascii=False)};\n'
              'process.stdout.write(JSON.stringify(casos.map(([r, a, x]) => _exportado.puede(r, a, x))));')
    ok, salida = agente.ejecutar_node(prueba)
    if not ok:
        return [f'el módulo da error al ejecutarlo: {salida}']
    problemas = []
    for (rol, accion, recurso), obtenido in zip(casos, json.loads(salida)):
        esperado = accion in tabla.get(rol, {}).get(recurso, set())
        if obtenido is not esperado:
            problemas.append(f"puede('{rol}', '{accion}', '{recurso}') da {json.dumps(obtenido)}, "
                             f'tenía que dar {json.dumps(esperado)}')
    return problemas

Overwriting permiso/traductor_permiso.py


In [6]:
%%file permiso/main.py
"""Uso: python main.py <archivo.permiso>
Traduce el programa y escribe <archivo.permiso>.js. Si hay errores, los muestra y no genera el archivo."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
from parser_permiso import ErrorPermiso
from agente_permiso import ErrorAgente
from traductor_permiso import traducir

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')  # tildes y símbolos en la consola de Windows

if len(sys.argv) != 2:
    print(__doc__)
    sys.exit(2)

fuente = Path(sys.argv[1])
try:
    resultado = traducir(fuente.read_text(encoding='utf-8'), avisar=print, origen=fuente.name)
except ErrorPermiso as e:
    print(e)
    sys.exit(1)
except ErrorAgente as e:
    print(f'Error de generación: {e}')
    sys.exit(1)
except ConnectionError:
    print('No se pudo conectar con ollama: abrilo (y descargá el modelo con "ollama pull llama3.2") y volvé a intentar.')
    sys.exit(1)

destino = fuente.with_name(fuente.name + '.js')
destino.write_text(resultado, encoding='utf-8')
print(f'Generado: {destino}')

Overwriting permiso/main.py


### **Ejecuciones**

Programas de ejemplo (se escriben como archivos `.permiso`):

In [7]:
%%file permiso/ejemplos/01_minimo.permiso
permisos 'Blog'
  rol 'lector'
    puede ver 'articulos'

Overwriting permiso/ejemplos/01_minimo.permiso


In [8]:
%%file permiso/ejemplos/02_tienda.permiso
# Permisos de la tienda online
permisos 'Tienda'
  rol 'invitado'
    puede ver 'productos'
  rol 'cliente' hereda 'invitado'
    puede crear 'pedidos'
    puede ver 'pedidos'
  rol 'admin' hereda 'cliente'
    puede todo 'productos'
    puede borrar 'pedidos'
  # el pasante ayuda al admin, pero no puede borrar nada
  rol 'pasante' hereda 'admin'
    no puede borrar 'productos'
    no puede borrar 'pedidos'

Overwriting permiso/ejemplos/02_tienda.permiso


In [9]:
%%file permiso/ejemplos/03_hospital.permiso
# Sistema de turnos e historias clínicas de un hospital
permisos 'Hospital'
  rol 'paciente'
    puede ver 'turnos'
    puede crear 'turnos'
    puede borrar 'turnos'
  rol 'recepcionista'
    puede todo 'turnos'
    puede ver 'pacientes'
    puede crear 'pacientes'
  rol 'enfermero' hereda 'recepcionista'
    puede ver 'historias clínicas'
  rol 'médico' hereda 'enfermero'
    puede crear 'historias clínicas'
    puede editar 'historias clínicas'
    puede crear 'recetas'
  # el residente trabaja como médico, pero todavía no firma recetas
  rol 'residente' hereda 'médico'
    no puede crear 'recetas'
  rol 'director' hereda 'médico'
    puede todo 'pacientes'
    puede ver 'estadísticas'

Overwriting permiso/ejemplos/03_hospital.permiso


In [10]:
%%file permiso/ejemplos/04_foro.permiso
# Foro de una comunidad
permisos 'Foro'
  rol 'visitante'
    puede ver 'temas'
    puede ver 'respuestas'
  rol 'miembro' hereda 'visitante'
    puede crear 'temas'
    puede crear 'respuestas'
  rol 'moderador' hereda 'miembro'
    puede editar 'temas'
    puede borrar 'temas'
    puede borrar 'respuestas'
  # el bot de spam solo puede borrar: no hereda de nadie
  rol 'bot'
    puede borrar 'respuestas'

Overwriting permiso/ejemplos/04_foro.permiso


In [11]:
%%file permiso/ejemplos/05_campus.permiso
permisos 'Campus virtual'
  rol 'alumno'
    puede ver 'materias'
    puede ver 'notas'
    puede crear 'entregas'
  rol 'ayudante' hereda 'alumno'
    puede ver 'entregas'
    puede editar 'notas'
  rol 'profesor' hereda 'ayudante'
    puede todo 'materias'
    puede todo 'notas'
    # las notas cerradas no se borran: todo menos borrar
    no puede borrar 'notas'
  rol 'administrativo'
    puede ver 'materias'
    puede todo 'inscripciones'
  # un alumno suspendido sigue viendo sus notas pero no puede entregar
  rol 'suspendido' hereda 'alumno'
    no puede crear 'entregas'

Overwriting permiso/ejemplos/05_campus.permiso


In [12]:
%%file permiso/ejemplos/06_error_lexico.permiso
permisos 'Blog'
  rol 'lector'
    puede leer 'articulos'

Overwriting permiso/ejemplos/06_error_lexico.permiso


In [13]:
%%file permiso/ejemplos/07_error_sintactico.permiso
permisos 'Tienda'
  rol 'cliente'
    puede ver 'productos'
    rol 'admin'
      puede todo 'productos'

Overwriting permiso/ejemplos/07_error_sintactico.permiso


In [14]:
%%file permiso/ejemplos/08_error_semantico.permiso
permisos 'Escuela'
  rol 'docente' hereda 'preceptor'
    puede editar 'notas'
  rol 'preceptor' hereda 'docente'
    puede ver 'asistencia'
  rol 'alumno' hereda 'tutor'
    puede ver 'notas'
  rol 'directivo'
    puede ver 'notas'
    no puede ver 'notas'

Overwriting permiso/ejemplos/08_error_semantico.permiso


In [15]:
import sys, subprocess, json, html, urllib.request, urllib.error
from pathlib import Path
from IPython.display import HTML, display

ACCIONES = ['ver', 'crear', 'editar', 'borrar']

def node(codigo):
    r = subprocess.run(['node', '-'], input=codigo, capture_output=True, text=True, encoding='utf-8')
    return r.stdout if r.returncode == 0 else r.stderr

def mostrar_tabla(modulo):
    """Carga el módulo generado con require() y le pregunta a puede() por cada rol, recurso y acción."""
    salida = node(f"""
        const m = require({json.dumps(str(modulo.resolve()))});
        const roles = Object.keys(m.PERMISOS);
        const recursos = [...new Set(roles.flatMap(r => Object.keys(m.PERMISOS[r].puede)))];
        const tabla = roles.map(r => [r, recursos.map(x => {json.dumps(ACCIONES)}.filter(a => m.puede(r, a, x)))]);
        process.stdout.write(JSON.stringify({{recursos, tabla}}));""")
    datos = json.loads(salida)
    filas = ''.join('<tr><th style="text-align:left">' + html.escape(rol) + '</th>' +
                    ''.join('<td style="text-align:left">' + (', '.join(acc) or '—') + '</td>' for acc in acciones) + '</tr>'
                    for rol, acciones in datos['tabla'])
    cabecera = ''.join(f'<th style="text-align:left">{html.escape(x)}</th>' for x in datos['recursos'])
    display(HTML(f'<p><b>Qué puede hacer cada rol</b> (respuestas de <code>puede()</code>, con la herencia aplicada):</p>'
                 f'<table><tr><th>rol</th>{cabecera}</tr>{filas}</table>'))

def probar_api(modulo, pedidos, puerto=3457):
    """Levanta el servidor Express de la tienda con el módulo generado, le hace pedidos HTTP reales y lo apaga.
    Cada pedido es (rol o None, método, ruta)."""
    if not Path('node_modules/express').exists():
        subprocess.run('npm install express --no-fund --no-audit', shell=True, check=True)
    servidor = subprocess.Popen(['node', 'permiso/servidor_tienda.js', str(modulo), str(puerto)],
                                stdout=subprocess.PIPE, text=True, encoding='utf-8')
    try:
        linea = servidor.stdout.readline().strip()
        if 'escuchando' not in linea:
            raise RuntimeError(f'El servidor no arrancó (¿el puerto {puerto} está ocupado?): {linea}')
        print(linea, '\n')
        for rol, metodo, ruta in pedidos:
            pedido = urllib.request.Request(f'http://127.0.0.1:{puerto}{ruta}', method=metodo,
                                            headers={'X-Rol': rol} if rol else {})
            try:
                with urllib.request.urlopen(pedido) as r:
                    estado, cuerpo = r.status, r.read().decode('utf-8')
            except urllib.error.HTTPError as e:
                estado, cuerpo = e.code, e.read().decode('utf-8')
            print(f"{rol or 'sin sesión':<11}{metodo:<8}{ruta:<16}→ {estado} {cuerpo}")
    finally:
        servidor.terminate()

def titulo(texto):
    print(f'\n───── {texto} ' + '─' * max(0, 60 - len(texto)))

def traducir_archivo(ruta):
    """Muestra el programa de entrada, ejecuta el traductor por consola y, si no hubo errores,
    muestra el módulo JavaScript generado y su tabla de permisos."""
    fuente = Path(ruta)
    titulo(f'Programa de entrada ({fuente.name})')
    print(fuente.read_text(encoding='utf-8').rstrip())
    titulo('Traducción')
    r = subprocess.run([sys.executable, 'permiso/main.py', ruta], capture_output=True, text=True, encoding='utf-8')
    print((r.stdout + r.stderr).strip())
    if r.returncode == 0:
        modulo = fuente.with_name(fuente.name + '.js')
        titulo(f'Código generado ({modulo.name})')
        print(modulo.read_text(encoding='utf-8').rstrip())
        mostrar_tabla(modulo)

**Ejemplo simple** (`01_minimo.permiso`):

In [16]:
traducir_archivo('permiso/ejemplos/01_minimo.permiso')


───── Programa de entrada (01_minimo.permiso) ─────────────────────
permisos 'Blog'
  rol 'lector'
    puede ver 'articulos'

───── Traducción ──────────────────────────────────────────────────


permisos 'Blog'
  rol 'lector'
  función contiene
  función puede
  middleware exigir
  verificación final con Node
Generado: permiso\ejemplos\01_minimo.permiso.js

───── Código generado (01_minimo.permiso.js) ──────────────────────
// Permisos de 'Blog': generado por el traductor permiso a partir de 01_minimo.permiso.
// Uso en Express: const { exigir } = require('./este-archivo');
//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);

const PERMISOS = {
  'lector': { hereda: null, puede: { 'articulos': ['ver'] } },
};

function contiene(lista, accion) {
  if (lista === undefined || !Array.isArray(lista)) return false;
  if (lista.includes(accion)) return true;
  if (lista.includes('todo')) return true;
  return false;
}

function puede(rol, accion, recurso) {
  if (!PERMISOS[rol]) return false;
  if (PERMISOS[rol].noPuede && contiene(PERMISOS[rol].noPuede[recurso], accion)) return false;
  if (contiene(PERMISOS[rol].puede[recurso], accion)) return true;
  retur

rol,articulos
lector,ver


**Ejemplo complejo** (`02_tienda.permiso`):

In [17]:
traducir_archivo('permiso/ejemplos/02_tienda.permiso')


───── Programa de entrada (02_tienda.permiso) ─────────────────────
# Permisos de la tienda online
permisos 'Tienda'
  rol 'invitado'
    puede ver 'productos'
  rol 'cliente' hereda 'invitado'
    puede crear 'pedidos'
    puede ver 'pedidos'
  rol 'admin' hereda 'cliente'
    puede todo 'productos'
    puede borrar 'pedidos'
  # el pasante ayuda al admin, pero no puede borrar nada
  rol 'pasante' hereda 'admin'
    no puede borrar 'productos'
    no puede borrar 'pedidos'

───── Traducción ──────────────────────────────────────────────────


permisos 'Tienda'
  rol 'invitado'
  rol 'cliente' (hereda de 'invitado')
  rol 'admin' (hereda de 'cliente')
  rol 'pasante' (hereda de 'admin') con 2 excepciones
  función contiene
  función puede
  middleware exigir
  verificación final con Node
Generado: permiso\ejemplos\02_tienda.permiso.js

───── Código generado (02_tienda.permiso.js) ──────────────────────
// Permisos de 'Tienda': generado por el traductor permiso a partir de 02_tienda.permiso.
// Uso en Express: const { exigir } = require('./este-archivo');
//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);

const PERMISOS = {
  'invitado': { hereda: null, puede: { productos: ['ver'] } },
  'cliente': { hereda: 'invitado', puede: { 'pedidos': ['crear', 'ver'] } },
  admin: { hereda: 'cliente', puede: { productos: ['todo'], pedidos: ['borrar'] } },
  'pasante': { hereda: 'admin', puede: {}, noPuede: { productos: ['borrar'], pedidos: ['borrar'] } },
};

function contiene(lista, accion) {
  if (lista === 

rol,productos,pedidos
invitado,ver,—
cliente,ver,"ver, crear"
admin,"ver, crear, editar, borrar","ver, crear, borrar"
pasante,"ver, crear, editar","ver, crear"


**La tienda funcionando en una API real.** El servidor Express `permiso/servidor_tienda.js` protege cada ruta con `exigir(acción, recurso)` usando el módulo que acaba de generar el traductor. El inicio de sesión se simula con el encabezado `X-Rol` (en una aplicación real lo pondría el login). Se le hacen pedidos HTTP de verdad y se muestra el código de estado que responde:

In [18]:
%%file permiso/servidor_tienda.js
// API de ejemplo de la tienda, protegida con el módulo de permisos que generó el traductor permiso.
// Uso: node servidor_tienda.js <módulo generado .js> [puerto]
const path = require('path');
const express = require('express');
const { exigir } = require(path.resolve(process.argv[2]));

const app = express();
const puerto = Number(process.argv[3] || 3457);

// Inicio de sesión simulado: el rol llega en el encabezado X-Rol.
// En una aplicación real, esto lo resuelve el login (por ejemplo, con una cookie de sesión o un token).
app.use((req, res, next) => {
  const rol = req.get('X-Rol');
  if (rol) req.usuario = { rol };
  next();
});

// Cada ruta exige un permiso: acción + recurso, como en el programa en permiso.
app.get('/productos',       exigir('ver', 'productos'),    (req, res) => res.json(['mate', 'termo', 'yerba']));
app.post('/productos',      exigir('crear', 'productos'),  (req, res) => res.status(201).send('Producto creado'));
app.put('/productos/:id',   exigir('editar', 'productos'), (req, res) => res.send(`Producto ${req.params.id} editado`));
app.delete('/productos/:id', exigir('borrar', 'productos'), (req, res) => res.send(`Producto ${req.params.id} borrado`));
app.get('/pedidos',         exigir('ver', 'pedidos'),      (req, res) => res.json([{ id: 3, total: 4500 }]));
app.post('/pedidos',        exigir('crear', 'pedidos'),    (req, res) => res.status(201).send('Pedido creado'));
app.delete('/pedidos/:id',  exigir('borrar', 'pedidos'),   (req, res) => res.send(`Pedido ${req.params.id} borrado`));

// Solo escucha en esta computadora (127.0.0.1).
app.listen(puerto, '127.0.0.1', () => console.log(`Tienda escuchando en http://127.0.0.1:${puerto}`));

Overwriting permiso/servidor_tienda.js


In [19]:
probar_api('permiso/ejemplos/02_tienda.permiso.js', [
    (None,        'GET',    '/productos'),     # sin sesión → 401
    ('invitado',  'GET',    '/productos'),
    ('invitado',  'POST',   '/pedidos'),       # el invitado no hace pedidos → 403
    ('cliente',   'POST',   '/pedidos'),
    ('cliente',   'PUT',    '/productos/7'),
    ('admin',     'DELETE', '/productos/7'),
    ('pasante',   'PUT',    '/productos/7'),   # heredado del admin
    ('pasante',   'DELETE', '/productos/7'),   # excepción: no puede borrar → 403
    ('pasante',   'DELETE', '/pedidos/3'),     # excepción → 403
])

Tienda escuchando en http://127.0.0.1:3457 

sin sesión GET     /productos      → 401 Falta iniciar sesión
invitado   GET     /productos      → 200 ["mate","termo","yerba"]
invitado   POST    /pedidos        → 403 Sin permiso
cliente    POST    /pedidos        → 201 Pedido creado
cliente    PUT     /productos/7    → 403 Sin permiso
admin      DELETE  /productos/7    → 200 Producto 7 borrado
pasante    PUT     /productos/7    → 200 Producto 7 editado
pasante    DELETE  /productos/7    → 403 Sin permiso
pasante    DELETE  /pedidos/3      → 403 Sin permiso


**Otros ejemplos:**

In [20]:
traducir_archivo('permiso/ejemplos/03_hospital.permiso')


───── Programa de entrada (03_hospital.permiso) ───────────────────
# Sistema de turnos e historias clínicas de un hospital
permisos 'Hospital'
  rol 'paciente'
    puede ver 'turnos'
    puede crear 'turnos'
    puede borrar 'turnos'
  rol 'recepcionista'
    puede todo 'turnos'
    puede ver 'pacientes'
    puede crear 'pacientes'
  rol 'enfermero' hereda 'recepcionista'
    puede ver 'historias clínicas'
  rol 'médico' hereda 'enfermero'
    puede crear 'historias clínicas'
    puede editar 'historias clínicas'
    puede crear 'recetas'
  # el residente trabaja como médico, pero todavía no firma recetas
  rol 'residente' hereda 'médico'
    no puede crear 'recetas'
  rol 'director' hereda 'médico'
    puede todo 'pacientes'
    puede ver 'estadísticas'

───── Traducción ──────────────────────────────────────────────────


permisos 'Hospital'
  rol 'paciente'
  rol 'recepcionista'
  rol 'enfermero' (hereda de 'recepcionista')
  rol 'médico' (hereda de 'enfermero')
  rol 'residente' (hereda de 'médico') con 1 excepción
  rol 'director' (hereda de 'médico')
  función contiene
  función puede
  middleware exigir
  verificación final con Node
Generado: permiso\ejemplos\03_hospital.permiso.js

───── Código generado (03_hospital.permiso.js) ────────────────────
// Permisos de 'Hospital': generado por el traductor permiso a partir de 03_hospital.permiso.
// Uso en Express: const { exigir } = require('./este-archivo');
//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);

const PERMISOS = {
  'paciente': { hereda: null, puede: { turnos: ['ver', 'crear', 'borrar'] } },
  'recepcionista': { hereda: null, puede: { turnos: ['todo'], pacientes: ['ver', 'crear'] } },
  enfermero: { hereda: 'recepcionista', puede: { 'historias clínicas': ['ver'] } },
  'médico': { hereda: 'enfermero', puede: { '

rol,turnos,pacientes,historias clínicas,recetas,estadísticas
paciente,"ver, crear, borrar",—,—,—,—
recepcionista,"ver, crear, editar, borrar","ver, crear",—,—,—
enfermero,"ver, crear, editar, borrar","ver, crear",ver,—,—
médico,"ver, crear, editar, borrar","ver, crear","ver, crear, editar",crear,—
residente,"ver, crear, editar, borrar","ver, crear","ver, crear, editar",—,—
director,"ver, crear, editar, borrar","ver, crear, editar, borrar","ver, crear, editar",crear,ver


In [21]:
traducir_archivo('permiso/ejemplos/04_foro.permiso')


───── Programa de entrada (04_foro.permiso) ───────────────────────
# Foro de una comunidad
permisos 'Foro'
  rol 'visitante'
    puede ver 'temas'
    puede ver 'respuestas'
  rol 'miembro' hereda 'visitante'
    puede crear 'temas'
    puede crear 'respuestas'
  rol 'moderador' hereda 'miembro'
    puede editar 'temas'
    puede borrar 'temas'
    puede borrar 'respuestas'
  # el bot de spam solo puede borrar: no hereda de nadie
  rol 'bot'
    puede borrar 'respuestas'

───── Traducción ──────────────────────────────────────────────────


permisos 'Foro'
  rol 'visitante'
  rol 'miembro' (hereda de 'visitante')
  rol 'moderador' (hereda de 'miembro')
  rol 'bot'
  función contiene
  función puede
  middleware exigir
  verificación final con Node
Generado: permiso\ejemplos\04_foro.permiso.js

───── Código generado (04_foro.permiso.js) ────────────────────────
// Permisos de 'Foro': generado por el traductor permiso a partir de 04_foro.permiso.
// Uso en Express: const { exigir } = require('./este-archivo');
//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);

const PERMISOS = {
  visitante: { hereda: null, puede: { temas: ['ver'], respuestas: ['ver'] } },
  miembro: { hereda: 'visitante', puede: { temas: ['crear'], respuestas: ['crear'] } },
  'moderador': { hereda: 'miembro', puede: { temas: ['editar', 'borrar'], respuestas: ['borrar'] } },
  bot: { hereda: null, puede: { respuestas: ['borrar'] } },
};

function contiene(lista, accion) {
  if (lista === undefined || !Array.isArray(lista)) return

rol,temas,respuestas
visitante,ver,ver
miembro,"ver, crear","ver, crear"
moderador,"ver, crear, editar, borrar","ver, crear, borrar"
bot,—,borrar


In [22]:
traducir_archivo('permiso/ejemplos/05_campus.permiso')


───── Programa de entrada (05_campus.permiso) ─────────────────────
permisos 'Campus virtual'
  rol 'alumno'
    puede ver 'materias'
    puede ver 'notas'
    puede crear 'entregas'
  rol 'ayudante' hereda 'alumno'
    puede ver 'entregas'
    puede editar 'notas'
  rol 'profesor' hereda 'ayudante'
    puede todo 'materias'
    puede todo 'notas'
    # las notas cerradas no se borran: todo menos borrar
    no puede borrar 'notas'
  rol 'administrativo'
    puede ver 'materias'
    puede todo 'inscripciones'
  # un alumno suspendido sigue viendo sus notas pero no puede entregar
  rol 'suspendido' hereda 'alumno'
    no puede crear 'entregas'

───── Traducción ──────────────────────────────────────────────────


permisos 'Campus virtual'
  rol 'alumno'
  rol 'ayudante' (hereda de 'alumno')
  rol 'profesor' (hereda de 'ayudante') con 1 excepción
  rol 'administrativo'
  rol 'suspendido' (hereda de 'alumno') con 1 excepción
  función contiene
  función puede
  middleware exigir
  verificación final con Node
Generado: permiso\ejemplos\05_campus.permiso.js

───── Código generado (05_campus.permiso.js) ──────────────────────
// Permisos de 'Campus virtual': generado por el traductor permiso a partir de 05_campus.permiso.
// Uso en Express: const { exigir } = require('./este-archivo');
//                 app.post('/ruta', exigir('crear', 'recurso'), manejador);

const PERMISOS = {
  'alumno': { hereda: null, puede: { materias: ['ver'], notas: ['ver'], entregas: ['crear'] } },
  'ayudante': { hereda: 'alumno', puede: { entregas: ['ver'], notas: ['editar'] } },
  'profesor': { hereda: 'ayudante', puede: { materias: ['todo'], notas: ['todo'] }, noPuede: { notas: ['borrar'] } },
  administrativo: { here

rol,materias,notas,entregas,inscripciones
alumno,ver,ver,crear,—
ayudante,ver,"ver, editar","ver, crear",—
profesor,"ver, crear, editar, borrar","ver, crear, editar","ver, crear",—
administrativo,ver,—,—,"ver, crear, editar, borrar"
suspendido,ver,ver,—,—


**Programas con errores** (no se genera el módulo):

In [23]:
traducir_archivo('permiso/ejemplos/06_error_lexico.permiso')


───── Programa de entrada (06_error_lexico.permiso) ───────────────
permisos 'Blog'
  rol 'lector'
    puede leer 'articulos'

───── Traducción ──────────────────────────────────────────────────


Error léxico:
  línea 3: Palabra desconocida: 'leer' (columna 11)


In [24]:
traducir_archivo('permiso/ejemplos/07_error_sintactico.permiso')


───── Programa de entrada (07_error_sintactico.permiso) ───────────
permisos 'Tienda'
  rol 'cliente'
    puede ver 'productos'
    rol 'admin'
      puede todo 'productos'

───── Traducción ──────────────────────────────────────────────────


Error sintáctico:
  línea 4: No se esperaba la palabra 'rol'


In [25]:
traducir_archivo('permiso/ejemplos/08_error_semantico.permiso')


───── Programa de entrada (08_error_semantico.permiso) ────────────
permisos 'Escuela'
  rol 'docente' hereda 'preceptor'
    puede editar 'notas'
  rol 'preceptor' hereda 'docente'
    puede ver 'asistencia'
  rol 'alumno' hereda 'tutor'
    puede ver 'notas'
  rol 'directivo'
    puede ver 'notas'
    no puede ver 'notas'

───── Traducción ──────────────────────────────────────────────────


Error semántico:
  línea 2: Herencia circular: 'docente' → 'preceptor' → 'docente'
  línea 6: El rol 'alumno' hereda de 'tutor', que no existe
  línea 10: Contradicción: el rol 'directivo' tiene "puede ver 'notas'" (línea 9) y "no puede ver 'notas'"


### **TestSuite**: (código + ejecuciones)

Casos de prueba automatizados con **pytest** (44 casos):

| Grupo | Casos | Qué se verifica |
| -- | -- | -- |
| Programas válidos | 10 | Los 5 ejemplos se traducen con el LLM y se **ejecuta el módulo generado con Node**: permisos propios, heredados en cadena, `todo`, excepciones (sobre lo heredado y "todo menos una acción" en el mismo rol), recursos con espacios y tildes, roles sin herencia; y para cada ejemplo, que **todas** las combinaciones rol × acción × recurso coincidan con la tabla esperada. No se compara el texto exacto porque la redacción del LLM puede variar. |
| Middleware de Express | 2 | Simulado: sin usuario → 401, sin permiso → 403, con permiso → `next()`. **Servidor Express real** con el módulo generado: pedidos HTTP que responden 401, 403, 201 y 200 según el rol (incluidas las excepciones del pasante) |
| Formato del módulo | 1 | Comentario inicial, `PERMISOS` (con `noPuede` cuando hay excepciones), una sola `contiene`, `puede` y `exigir`, `module.exports` al final |
| Tabla esperada | 2 | El cálculo del parser aplica herencia, `todo` y excepciones; la excepción se hereda y lo propio de un rol le gana a lo heredado |
| Errores léxicos | 6 | Sangría impar, tabulación, cadena sin cerrar, acción desconocida, comillas dobles, sangría que sube dos niveles |
| Errores sintácticos | 9 | Rol sin permisos, programa sin roles, permiso sin recurso, permiso sin acción, rol dentro de otro rol, falta el encabezado `permisos`, `hereda` sin nombre de rol, `no` sin `puede`, `no` después de `puede` |
| Errores semánticos | 12 | Rol repetido, herencia de un rol inexistente, herencia de sí mismo, herencia circular directa e indirecta, permiso repetido, contradicción (`puede ver` + `no puede ver`; permiso + `no puede todo`), excepción de más (sobre otro recurso; sobre algo no heredado), varios errores informados juntos, y casos válidos que **no** son error (`puede todo` + `no puede borrar`, `no puede todo` sobre lo heredado…) |
| Consola (`main.py`) | 2 | Caso feliz (genera el `.js` y se ejecuta sin errores) y caso con error (mensaje con la línea, no genera archivo) |

In [26]:
%%file permiso/test_permiso.py
"""TestSuite del lenguaje permiso (pytest).

- Programas válidos: se traducen con el LLM y se EJECUTA el módulo JavaScript generado con Node,
  verificando qué responde (no el texto exacto, porque la redacción del LLM puede variar).
- Middleware de Express: se simulan pedidos y se verifica que responda 401, 403 o deje pasar; y se levanta
  un servidor Express real con el módulo generado y se le hacen pedidos HTTP.
- Tabla esperada: el cálculo de permisos con herencia y excepciones que hace el parser.
- Errores léxicos, sintácticos y semánticos: se verifica el tipo de error y el mensaje.
- Programa completo por consola (main.py), como lo usaría un programador.
Ejecutar desde esta carpeta:  python -m pytest -v test_permiso.py
"""
import json
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

import pytest

CARPETA = Path(__file__).parent
sys.path.insert(0, str(CARPETA))
from parser_permiso import analizar, tabla_esperada, ErrorPermiso
from traductor_permiso import traducir
from agente_permiso import ejecutar_node

EJEMPLOS = CARPETA / 'ejemplos'


def leer(nombre):
    return (EJEMPLOS / nombre).read_text(encoding='utf-8')


def consultar(modulo, preguntas):
    """Ejecuta el módulo generado con Node y devuelve lo que responde puede() para cada (rol, acción, recurso)."""
    codigo = (modulo.replace('module.exports', 'const m') +
              f'\nprocess.stdout.write(JSON.stringify({json.dumps(preguntas, ensure_ascii=False)}'
              '.map(([r, a, x]) => m.puede(r, a, x))));')
    ok, salida = ejecutar_node(codigo)
    assert ok, salida
    return json.loads(salida)


def pedido_express(modulo, usuario, accion, recurso):
    """Simula un pedido a una ruta protegida con exigir(accion, recurso). Devuelve 'siguiente' o el estado HTTP."""
    req = json.dumps({'usuario': usuario} if usuario else {}, ensure_ascii=False)
    codigo = (modulo.replace('module.exports', 'const m') +
              f'\nlet r = null;\n'
              'const res = { status(n) { return { send() { r = n; } }; } };\n'
              f'm.exigir({json.dumps(accion)}, {json.dumps(recurso, ensure_ascii=False)})({req}, res, () => {{ r = "siguiente"; }});\n'
              'process.stdout.write(JSON.stringify(r));')
    ok, salida = ejecutar_node(codigo)
    assert ok, salida
    return json.loads(salida)


# ================================================================ programas válidos (con el LLM)

def test_minimo():
    m = traducir(leer('01_minimo.permiso'))
    assert consultar(m, [['lector', 'ver', 'articulos'], ['lector', 'editar', 'articulos'],
                         ['admin', 'ver', 'articulos']]) == [True, False, False]


def test_tienda_herencia_en_cadena_todo_y_excepciones():
    m = traducir(leer('02_tienda.permiso'))
    assert consultar(m, [
        ['invitado', 'ver', 'productos'],     # propio
        ['invitado', 'crear', 'pedidos'],     # no lo tiene
        ['cliente', 'ver', 'productos'],      # heredado de invitado
        ['admin', 'crear', 'pedidos'],        # heredado de cliente
        ['admin', 'editar', 'productos'],     # 'todo' incluye editar
        ['cliente', 'borrar', 'pedidos'],     # solo lo tiene admin
        ['pasante', 'editar', 'productos'],   # heredado de admin
        ['pasante', 'borrar', 'productos'],   # excepción: no puede, aunque admin sí
        ['pasante', 'borrar', 'pedidos'],     # excepción
    ]) == [True, False, True, True, True, False, True, False, False]


def test_hospital_recursos_con_espacios_y_tildes():
    m = traducir(leer('03_hospital.permiso'))
    assert consultar(m, [
        ['director', 'ver', 'historias clínicas'],   # director → médico → enfermero
        ['director', 'borrar', 'turnos'],            # ... → recepcionista: 'todo' sobre turnos
        ['enfermero', 'editar', 'historias clínicas'],
        ['paciente', 'editar', 'turnos'],
        ['residente', 'crear', 'recetas'],           # excepción sobre lo heredado de médico
        ['residente', 'editar', 'historias clínicas'],
        ['director', 'crear', 'recetas'],            # director hereda de médico, no de residente
    ]) == [True, True, False, False, False, True, True]


def test_foro_rol_sin_herencia():
    m = traducir(leer('04_foro.permiso'))
    assert consultar(m, [['bot', 'borrar', 'respuestas'], ['bot', 'ver', 'temas'],
                         ['moderador', 'ver', 'respuestas']]) == [True, False, True]


def test_campus_todo_menos_una_accion_en_el_mismo_rol():
    m = traducir(leer('05_campus.permiso'))
    assert consultar(m, [['profesor', 'editar', 'notas'], ['profesor', 'borrar', 'notas'],
                         ['profesor', 'borrar', 'materias'], ['suspendido', 'ver', 'notas'],
                         ['suspendido', 'crear', 'entregas']]) == [True, False, True, True, False]


@pytest.mark.parametrize('ejemplo', ['01_minimo', '02_tienda', '03_hospital', '04_foro', '05_campus'])
def test_todas_las_combinaciones_coinciden_con_la_tabla(ejemplo):
    texto = leer(f'{ejemplo}.permiso')
    tabla = tabla_esperada(analizar(texto))
    recursos = sorted({x for permisos in tabla.values() for x in permisos})
    preguntas = [[r, a, x] for r in tabla for a in ('ver', 'crear', 'editar', 'borrar') for x in recursos]
    esperado = [a in tabla[r].get(x, set()) for r, a, x in preguntas]
    assert consultar(traducir(texto), preguntas) == esperado


# ================================================================ middleware de Express

def test_middleware_401_403_y_siguiente():
    m = traducir(leer('02_tienda.permiso'))
    assert pedido_express(m, None, 'crear', 'pedidos') == 401
    assert pedido_express(m, {'rol': 'invitado'}, 'crear', 'pedidos') == 403
    assert pedido_express(m, {'rol': 'cliente'}, 'crear', 'pedidos') == 'siguiente'
    assert pedido_express(m, {'rol': 'admin'}, 'borrar', 'productos') == 'siguiente'


def test_modulo_formateado():
    m = traducir(leer('02_tienda.permiso'))
    assert m.startswith("// Permisos de 'Tienda'")
    assert 'const PERMISOS = {' in m and m.rstrip().endswith('module.exports = { PERMISOS, puede, exigir };')
    assert m.count('function contiene(') == 1 and m.count('function puede(') == 1 and m.count('function exigir(') == 1
    assert "noPuede" in m                                   # el pasante tiene excepciones


def servidor_disponible():
    ok, _ = ejecutar_node(f"require.resolve('express', {{ paths: [{json.dumps(str(CARPETA))}] }});")
    return ok


@pytest.mark.skipif(not servidor_disponible(), reason='Express no está instalado (npm install express)')
def test_servidor_express_real(tmp_path):
    modulo = tmp_path / 'tienda.permiso.js'
    modulo.write_text(traducir(leer('02_tienda.permiso')), encoding='utf-8')
    puerto = '3471'
    servidor = subprocess.Popen(['node', str(CARPETA / 'servidor_tienda.js'), str(modulo), puerto],
                                stdout=subprocess.PIPE, text=True, encoding='utf-8')
    try:
        assert 'escuchando' in servidor.stdout.readline()

        def estado(metodo, ruta, rol=None):
            pedido = urllib.request.Request(f'http://127.0.0.1:{puerto}{ruta}', method=metodo,
                                            headers={'X-Rol': rol} if rol else {})
            try:
                with urllib.request.urlopen(pedido) as r:
                    return r.status
            except urllib.error.HTTPError as e:
                return e.code

        assert estado('GET', '/productos') == 401
        assert estado('POST', '/pedidos', 'invitado') == 403
        assert estado('POST', '/pedidos', 'cliente') == 201
        assert estado('DELETE', '/productos/7', 'admin') == 200
        assert estado('DELETE', '/productos/7', 'pasante') == 403
        assert estado('PUT', '/productos/7', 'pasante') == 200
    finally:
        servidor.terminate()


# ================================================================ tabla esperada (parser, sin LLM)

def test_tabla_esperada_aplica_herencia_todo_y_excepciones():
    tabla = tabla_esperada(analizar(leer('02_tienda.permiso')))
    assert tabla['invitado'] == {'productos': {'ver'}}
    assert tabla['cliente'] == {'productos': {'ver'}, 'pedidos': {'crear', 'ver'}}
    assert tabla['admin'] == {'productos': {'ver', 'crear', 'editar', 'borrar'}, 'pedidos': {'crear', 'ver', 'borrar'}}
    assert tabla['pasante'] == {'productos': {'ver', 'crear', 'editar'}, 'pedidos': {'crear', 'ver'}}


def test_la_excepcion_se_hereda_y_lo_propio_gana():
    tabla = tabla_esperada(analizar(
        "permisos 'App'\n"
        "  rol 'a'\n    puede todo 'x'\n"
        "  rol 'b' hereda 'a'\n    no puede borrar 'x'\n"
        "  rol 'c' hereda 'b'\n    puede ver 'y'\n"            # hereda la excepción de b
        "  rol 'd' hereda 'b'\n    puede borrar 'x'\n"))        # lo propio de d le gana a la excepción de b
    assert tabla['c'] == {'x': {'ver', 'crear', 'editar'}, 'y': {'ver'}}
    assert tabla['d'] == {'x': {'ver', 'crear', 'editar', 'borrar'}}


# ================================================================ errores léxicos

E = "permisos 'App'\n  rol 'a'\n"


@pytest.mark.parametrize('programa, mensaje', [
    (E + "     puede ver 'x'\n", 'Sangría de 5 espacios'),
    (E + "    puede\tver 'x'\n", 'No se permiten tabulaciones'),
    (E + "    puede ver 'x\n", 'Cadena sin cerrar'),
    (E + "    puede leer 'x'\n", "Palabra desconocida: 'leer'"),
    (E + "    puede ver \"x\"\n", "Carácter no reconocido: '\"'"),
    ("permisos 'App'\n  rol 'a'\n      puede ver 'x'\n", 'sube más de un nivel'),
], ids=['sangria_impar', 'tabulacion', 'cadena_sin_cerrar', 'accion_desconocida', 'comillas_dobles', 'salto_de_nivel'])
def test_errores_lexicos(programa, mensaje):
    with pytest.raises(ErrorPermiso) as e:
        analizar(programa)
    assert e.value.tipo == 'léxico'
    assert any(mensaje in m for _, m in e.value.errores)


# ================================================================ errores sintácticos

@pytest.mark.parametrize('programa, linea', [
    (E, 3),                                                           # rol sin permisos
    ("permisos 'App'\n", None),                                       # programa sin roles
    (E + "    puede ver\n", 3),                                       # falta el recurso
    (E + "    puede 'x'\n", 3),                                       # falta la acción
    (E + "    puede ver 'x'\n    rol 'b'\n      puede ver 'x'\n", 4),  # rol dentro de un rol
    ("rol 'a'\n  puede ver 'x'\n", 1),                                # falta el encabezado permisos
    ("permisos 'App'\n  rol 'a' hereda\n    puede ver 'x'\n", 2),     # hereda sin el nombre del rol
    (E + "    no ver 'x'\n", 3),                                      # excepción sin 'puede'
    (E + "    puede no ver 'x'\n", 3),                                # 'no' en otro lugar
], ids=['rol_sin_permisos', 'sin_roles', 'falta_recurso', 'falta_accion', 'rol_dentro_de_rol', 'sin_encabezado',
        'hereda_sin_nombre', 'no_sin_puede', 'no_despues_de_puede'])
def test_errores_sintacticos(programa, linea):
    with pytest.raises(ErrorPermiso) as e:
        analizar(programa)
    assert e.value.tipo == 'sintáctico'
    assert e.value.errores[0][0] == linea


# ================================================================ errores semánticos

def rol(nombre, hereda=None, permiso="puede ver 'x'"):
    return f"  rol '{nombre}'" + (f" hereda '{hereda}'" if hereda else '') + f"\n    {permiso}\n"


@pytest.mark.parametrize('roles, mensaje', [
    (rol('a') + rol('a'), "El rol 'a' ya se definió en la línea 2"),
    (rol('a', 'z'), "hereda de 'z', que no existe"),
    (rol('a', 'a'), 'no puede heredar de sí mismo'),
    (rol('a', 'b') + rol('b', 'a'), "Herencia circular: 'a' → 'b' → 'a'"),
    (rol('a', 'c') + rol('b', 'a') + rol('c', 'b'), "Herencia circular: 'a' → 'c' → 'b' → 'a'"),
    ("  rol 'a'\n    puede ver 'x'\n    puede ver 'x'\n", 'ya está en la línea 3'),
    ("  rol 'a'\n    puede ver 'x'\n    no puede ver 'x'\n", 'Contradicción'),
    ("  rol 'a'\n    puede ver 'x'\n    no puede todo 'x'\n", 'Contradicción'),
    (rol('a', permiso="puede todo 'x'") + rol('b', 'a', "no puede ver 'y'"), 'no hace falta'),
    (rol('a') + rol('b', 'a', "no puede borrar 'x'"), 'no hace falta'),
], ids=['rol_repetido', 'hereda_inexistente', 'hereda_de_si_mismo', 'circular_directa', 'circular_indirecta',
        'permiso_repetido', 'contradiccion', 'contradiccion_con_todo', 'excepcion_sobre_otro_recurso',
        'excepcion_de_algo_no_heredado'])
def test_errores_semanticos(roles, mensaje):
    with pytest.raises(ErrorPermiso) as e:
        analizar("permisos 'App'\n" + roles)
    assert e.value.tipo == 'semántico'
    assert any(mensaje in m for _, m in e.value.errores)


def test_varios_errores_semanticos_se_informan_juntos():
    with pytest.raises(ErrorPermiso) as e:
        analizar(leer('08_error_semantico.permiso'))
    assert [l for l, _ in e.value.errores] == [2, 6, 10]


def test_casos_validos_que_no_son_error():
    # mismo recurso con acciones distintas
    assert len(analizar("permisos 'App'\n  rol 'a'\n    puede ver 'x'\n    puede crear 'x'\n")['roles'][0]['permisos']) == 2
    # 'todo' menos una acción, en el mismo rol
    analizar("permisos 'App'\n  rol 'a'\n    puede todo 'x'\n    no puede borrar 'x'\n")
    # 'no puede todo' sobre algo heredado: el rol pierde el recurso entero
    analizar("permisos 'App'\n  rol 'a'\n    puede ver 'x'\n  rol 'b' hereda 'a'\n    no puede todo 'x'\n")


# ================================================================ programa por consola (main.py)

def ejecutar_main(tmp_path, programa):
    fuente = tmp_path / 'prueba.permiso'
    fuente.write_text(programa, encoding='utf-8')
    r = subprocess.run([sys.executable, str(CARPETA / 'main.py'), str(fuente)],
                       capture_output=True, text=True, encoding='utf-8')
    return r, tmp_path / 'prueba.permiso.js'


def test_main_caso_feliz(tmp_path):
    r, salida = ejecutar_main(tmp_path, leer('01_minimo.permiso'))
    assert r.returncode == 0
    assert salida.exists()
    ok, _ = ejecutar_node(salida.read_text(encoding='utf-8'))   # el archivo generado se ejecuta sin errores
    assert ok


def test_main_caso_con_error(tmp_path):
    r, salida = ejecutar_main(tmp_path, leer('08_error_semantico.permiso'))
    assert r.returncode == 1
    assert 'Error semántico' in r.stdout and 'línea 2' in r.stdout and 'Herencia circular' in r.stdout
    assert not salida.exists()

Overwriting permiso/test_permiso.py


In [27]:
!"{sys.executable}" -m pytest -v --color=no -p no:cacheprovider permiso/test_permiso.py

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Franco\venv-planilla\Scripts\python.exe
rootdir: C:\Users\Franco\Desktop\TP permiso
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 44 items

permiso/test_permiso.py::test_minimo PASSED                              [  2%]
permiso/test_permiso.py::test_tienda_herencia_en_cadena_todo_y_excepciones PASSED [  4%]
permiso/test_permiso.py::test_hospital_recursos_con_espacios_y_tildes PASSED [  6%]
permiso/test_permiso.py::test_foro_rol_sin_herencia PASSED               [  9%]
permiso/test_permiso.py::test_campus_todo_menos_una_accion_en_el_mismo_rol PASSED [ 11%]
permiso/test_permiso.py::test_todas_las_combinaciones_coinciden_con_la_tabla[01_minimo] PASSED [ 13%]
permiso/test_permiso.py::test_todas_las_combinaciones_coinciden_con_la_tabla[02_tienda] PASSED [ 15%]
permiso/test_permiso.py::test_todas_las_combinaciones_coinc

### **Conclusiones**:

* Se diseñó e implementó **permiso**, un lenguaje de dominio específico para describir en español los roles y permisos de una aplicación web, con una gramática de producciones recursivas que admite infinitos programas fuente y una estructura marcada por sangría. Su salida es un módulo JavaScript que se usa directamente como middleware en un backend Express, y se probó en un servidor Express real.
* **La sangría no es un problema regular:** para saber si una línea abre o cierra un bloque hay que recordar los niveles anteriores, algo que un autómata finito no puede hacer. Por eso el lexer, además de las expresiones regulares de PLY, usa una pila de niveles que genera los tokens `INDENTA` y `DESINDENTA`; con ellos, la gramática trata la sangría como si fueran llaves.
* **La herencia circular no se puede detectar con la gramática:** `rol 'a' hereda 'b'` es sintácticamente correcto aunque `b` herede de `a`. Es un control semántico que necesita conocer todos los roles, por eso se hace al reducir `<Programa>`, siguiendo la cadena `hereda` de cada rol. Si no se detectara, la función `puede` generada entraría en un ciclo infinito.
* **Las excepciones obligaron a definir una regla de decisión** (lo más cercano al rol gana; dentro de un rol, la excepción gana) y trajeron controles semánticos que tampoco puede hacer la gramática: una contradicción solo se ve comparando las líneas de un rol, y una excepción de más solo se detecta recorriendo la herencia para saber si el rol tendría ese permiso.
* **Separar el análisis de la generación** fue clave: el parser arma un árbol y verifica la semántica antes de llamar al LLM. Así los errores se informan completos, con su línea, y no se hacen pedidos al modelo para programas incorrectos. Además, como el parser conoce la tabla de permisos completa, puede **comprobar el código del LLM ejecutándolo**, sin que el usuario tenga que escribir casos de prueba.
* **El LLM no es confiable por sí solo.** En las pruebas, al pedirle la función `puede`, copiaba la tabla de ejemplo del pedido adentro de la función (tapando la tabla real), declaraba constantes de más, interpretaba `todo` como si fuera la acción pedida en lugar de un permiso guardado, y en la tabla omitía la coma entre entradas. Se resolvió:
  * describiendo cada construcción con su significado y los datos del parser, sin ejemplos que pudiera copiar, y aclarando lo que confundía (que `'todo'` es un valor guardado en el arreglo y no el parámetro `accion`, y que la función tiene que devolver siempre `true` o `false`);
  * quedándose solo con la función pedida y descartando lo que el LLM agrega alrededor (y las comillas, cuando devolvía el código como si fuera un texto);
  * **dividiendo la lógica en funciones chicas**: con las excepciones, al pedirle `puede` completa el modelo arreglaba `'todo'` en un lado y lo rompía en el otro, intento tras intento; separando `contiene` (resuelve `'todo'`) de `puede` (herencia y excepciones), las dos salieron bien al primer intento;
  * mostrando la **forma** de la respuesta con `...` para que la complete (por ejemplo `'pasante': { hereda: ..., puede: {}, noPuede: { ... } },`): sin eso, a veces omitía la clave del rol o le inventaba permisos a un rol que solo tenía excepciones;
  * con avisos **al estilo de un compilador**: si el código usa un nombre que no existe, se le sugiere el más parecido (el modelo escribía `acccion` en lugar de `accion` en todos los intentos, porque se parece a `acciones`; se cambió el nombre del parámetro a `lista` y se agregó la sugerencia);
  * **ejecutando** cada respuesta con Node y devolviéndole al LLM los casos que fallan **con la explicación basada en los datos** (por ejemplo, *"puede('alto', 'editar', 'notas') devolvió false y tenía que devolver true porque el arreglo de 'notas' del rol 'alto' es ["todo"]"*). Solo con el caso que falla, sin el porqué, el modelo repetía el mismo error en todos los intentos;
  * empezando una conversación nueva con otra semilla cuando una conversación se traba en el mismo error: aun con `temperature=0`, la respuesta del modelo cambió entre una sesión de ollama y otra;
  * usando `temperature=0` para que la salida sea lo más estable posible (y los tests también);
  * armando el módulo y su formato (comas, espaciado) con código propio, y verificando el módulo completo al final.
* **Limitaciones:** el traductor depende de que ollama y Node estén instalados (y ollama abierto); la primera traducción tarda más porque se carga el modelo; la tabla de permisos queda fija en el código (cambiarla requiere volver a traducir).
* **Posibles extensiones:** permisos con condiciones (*"puede editar sus propios pedidos"*), herencia múltiple (habría que definir qué pasa si un padre permite y otro tiene una excepción), y generar también una página HTML con la tabla de permisos para documentar la API.